# IBL neural processing — PETH, decoding, hesitation, and PCA

Downstream processing for one or more sessions and regions:

- QC label 1 units from one selected probe per region
- prior-balanced contralateral PETHs around stimulus and first movement
- pre/post-stimulus and pre/post-action Linear-SVM decoding
- hesitation scores from official wheel-velocity direction changes, with
  separate no-hesitation/hesitation comparisons, direct no-hesitation/Q4
  comparisons, and positive-score Q1–Q4 views
- three-component PCA trajectories through the same four 200 ms trial windows
- an additional time-resolved, three-component PCA from 200 ms before stimulus
  through the normalized stimulus-to-action interval to 200 ms after action
- a separate per-session QC figure for the raw hesitation scores and their
  exact Q1–Q4 assignment
- separate contrast, hesitation-state, hesitation-quartile, and
  no-hesitation/Q4 PETH and SVM figures, plus event-window and continuous
  PCA figures in both 2D and 3D

Probe selection is automatic and follows the same target-unit criterion as the selection
notebook.


## 0. Environment


In [ ]:
%pip install -q -U ONE-api ibllib scikit-learn


In [ ]:
import os

os.environ.setdefault('ONE_HTTP_DL_THREADS', '1')


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable
import gc
import re
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import proj3d
from IPython.display import display

from scipy.interpolate import PchipInterpolator

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

from one.api import ONE
from brainbox.behavior.wheel import direction_changes
from brainbox.io.one import SessionLoader, SpikeSortingLoader
from brainbox.singlecell import calculate_peths
from iblatlas.atlas import AllenAtlas
from iblatlas.regions import BrainRegions


ONE_PASSWORD = "international"

ONE.setup(base_url='https://openalyx.internationalbrainlab.org', silent=True)
one = ONE(password=ONE_PASSWORD)
print("Connected:", not one.offline)


## 1. User configuration

Edit `SESSION_IDS` and `REGIONS`. One or many values are accepted. The example EID is the
top-ranked session stored in the original notebook output; replace it with the shortlist
produced by the selection notebook.


In [ ]:
@dataclass(frozen=True)
class ProcessingConfig:
    session_ids: tuple[str, ...]
    regions: tuple[str, ...]
    atlas_mapping: str = "Beryl"
    accepted_histology: tuple[str, ...] = ("alf", "resolved")
    min_good_units_per_region: int = 5

    # Shared trial design
    expected_priors: tuple[float, ...] = (0.2, 0.5, 0.8)
    contrast_levels: tuple[float, ...] = (0.0, 0.0625, 0.125, 0.25, 1.0)
    movement_latency_s: tuple[float, float] = (0.080, 2.000)

    # Event-aligned population activity
    peth_pre_s: float = 0.400
    peth_post_s: float = 0.400
    peth_bin_s: float = 0.025
    peth_smoothing_s: float = 0.025
    display_gap_s: float = 0.200
    min_trials_per_peth_condition: int = 1

    # Choice decoder
    decoder_window_s: float = 0.200
    decoder_n_splits: int = 5
    random_state: int = 42
    decoder_max_iter: int = 1_000

    # Hesitation: official filtered wheel velocity and direction changes.
    # Scores are counted from stimulus onset to response time. Neural
    # windows remain exactly the PETH/SVM windows configured above.
    hesitation_n_quartiles: int = 4
    wheel_sampling_hz: float = 1_000.0
    wheel_corner_frequency_hz: float = 20.0
    wheel_filter_order: int = 8

    # Event-window PCA uses the same four decoder windows and valid
    # trials. Both choices share each 2D/3D axis; centroids are shown
    # by default, with optional bounded individual trajectories.
    pca_n_components: int = 3
    pca_show_individual_trials: bool = False
    pca_max_individual_trials_per_group: int = 8
    pca_individual_alpha: float = 0.045
    pca_view_elev: float = 24.0
    pca_view_azim: float = -58.0

    # Continuous PCA keeps three components but produces both a clear
    # PC1–PC2 view and a full PC1–PC2–PC3 view from the same fit.
    continuous_pca_n_components: int = 3
    continuous_pca_pre_stim_s: float = 0.200
    continuous_pca_post_action_s: float = 0.200
    continuous_pca_sample_step_s: float = 0.020
    continuous_pca_decision_points: int = 20
    continuous_pca_smoothing_s: float = 0.030
    continuous_pca_show_individual_trials: bool = False
    continuous_pca_max_individual_trials_per_group: int = 8
    continuous_pca_individual_alpha: float = 0.035

    # Publication figures are separate PNG files in analysis folders.
    figure_output_dir: str = "ibl_figures"
    figure_formats: tuple[str, ...] = ("png",)
    figure_dpi: int = 300

    continue_on_error: bool = False


# ------------------------------------------------------------------
# EDIT HERE
# ------------------------------------------------------------------
SESSION_IDS = [
    "a82800ce-f4e3-4464-9b80-4c3d6fade333",
]

REGIONS = ["LGd"]
# Example: REGIONS = ["LGd", "VISa", "CA1"]

CONFIG = ProcessingConfig(
    session_ids=tuple(SESSION_IDS),
    regions=tuple(REGIONS),
)

CONFIG


## 2. Data containers and validation


In [ ]:
REQUIRED_TRIAL_FIELDS = {
    "contrastLeft",
    "contrastRight",
    "probabilityLeft",
    "choice",
    "stimOn_times",
    "firstMovement_times",
    "response_times",
}


@dataclass
class NeuralSessionData:
    eid: str
    subject: str
    spike_times: np.ndarray
    spike_units: np.ndarray
    units: pd.DataFrame
    region_probes: pd.DataFrame
    probe_errors: pd.DataFrame


@dataclass
class PopulationPETH:
    time_s: np.ndarray
    mean_hz: np.ndarray
    sem_hz: np.ndarray
    n_units: int


@dataclass
class PCATrajectoryResult:
    scaler: StandardScaler
    model: PCA
    scores: pd.DataFrame
    centroids: pd.DataFrame
    explained_variance_ratio: np.ndarray
    n_trials: int
    n_units: int


@dataclass
class ContinuousPCATrajectoryResult:
    scaler: StandardScaler
    model: PCA
    scores: pd.DataFrame
    centroids: pd.DataFrame
    timepoints: pd.DataFrame
    movement_latencies_s: np.ndarray
    explained_variance_ratio: np.ndarray
    n_trials: int
    n_units: int
    n_timepoints: int


@dataclass
class HesitationResult:
    scores: np.ndarray
    quartiles: np.ndarray
    has_hesitation: np.ndarray
    direction_change_times: np.ndarray
    trial_table: pd.DataFrame
    summary: pd.DataFrame
    state_summary: pd.DataFrame
    no_hesitation_vs_q4: np.ndarray | None = None


@dataclass
class SessionAnalysis:
    eid: str
    subject: str
    unit_summary: pd.DataFrame
    region_probes: pd.DataFrame
    peth_results: dict[str, dict[str, dict[float, PopulationPETH]]]
    peth_condition_counts: pd.DataFrame
    peth_errors: pd.DataFrame
    hesitation: HesitationResult
    hesitation_peth_results: dict[
        str, dict[str, dict[int, PopulationPETH]]
    ]
    hesitation_peth_condition_counts: pd.DataFrame
    hesitation_peth_errors: pd.DataFrame
    hesitation_presence_peth_results: dict[
        str, dict[str, dict[int, PopulationPETH]]
    ]
    hesitation_presence_peth_condition_counts: pd.DataFrame
    hesitation_presence_peth_errors: pd.DataFrame
    decoder_results: dict[str, dict[str, pd.DataFrame]]
    decoder_summary: pd.DataFrame
    decoder_errors: pd.DataFrame
    pca_results: dict[str, PCATrajectoryResult]
    pca_summary: pd.DataFrame
    pca_errors: pd.DataFrame
    continuous_pca_results: dict[str, ContinuousPCATrajectoryResult]
    continuous_pca_summary: pd.DataFrame
    continuous_pca_errors: pd.DataFrame
    probe_errors: pd.DataFrame
    no_hesitation_vs_q4_peth_results: dict[
        str, dict[str, dict[int, PopulationPETH]]
    ] | None = None
    no_hesitation_vs_q4_peth_condition_counts: pd.DataFrame | None = None
    no_hesitation_vs_q4_peth_errors: pd.DataFrame | None = None


def validate_processing_config(config: ProcessingConfig) -> None:
    if not config.session_ids:
        raise ValueError("session_ids cannot be empty.")
    if not config.regions:
        raise ValueError("regions cannot be empty.")
    if len(set(config.session_ids)) != len(config.session_ids):
        raise ValueError("session_ids contains duplicates.")
    if len(set(config.regions)) != len(config.regions):
        raise ValueError("regions contains duplicates.")
    if config.peth_pre_s <= 0 or config.peth_post_s <= 0:
        raise ValueError("PETH windows must be positive.")
    if not np.isclose(config.peth_pre_s, config.peth_post_s):
        raise ValueError("The two-panel PETH plot requires symmetric windows.")
    if config.peth_bin_s <= 0 or config.peth_smoothing_s < 0:
        raise ValueError("Invalid PETH bin or smoothing value.")
    low, high = config.movement_latency_s
    if not (0 <= low < high):
        raise ValueError("movement_latency_s must satisfy 0 <= low < high.")
    if config.decoder_window_s <= 0:
        raise ValueError("decoder_window_s must be positive.")
    if config.decoder_n_splits < 2:
        raise ValueError("decoder_n_splits must be at least 2.")
    if config.hesitation_n_quartiles != 4:
        raise ValueError(
            "hesitation_n_quartiles must be 4 for the quartile figures."
        )
    if config.wheel_sampling_hz <= 0:
        raise ValueError("wheel_sampling_hz must be positive.")
    if not (
        0
        < config.wheel_corner_frequency_hz
        < config.wheel_sampling_hz / 2
    ):
        raise ValueError(
            "wheel_corner_frequency_hz must be between 0 and Nyquist."
        )
    if config.wheel_filter_order < 1:
        raise ValueError("wheel_filter_order must be positive.")
    if config.pca_n_components != 3:
        raise ValueError("pca_n_components must be 3 for the 3D trajectory plot.")
    if config.pca_max_individual_trials_per_group < 1:
        raise ValueError(
            "pca_max_individual_trials_per_group must be positive."
        )
    if not 0 <= config.pca_individual_alpha <= 1:
        raise ValueError("pca_individual_alpha must be between 0 and 1.")
    if not np.isfinite(config.pca_view_elev) or not np.isfinite(
        config.pca_view_azim
    ):
        raise ValueError("PCA view angles must be finite.")
    if config.continuous_pca_n_components != 3:
        raise ValueError(
            "continuous_pca_n_components must be 3 for the 2D/3D views."
        )
    if (
        config.continuous_pca_pre_stim_s <= 0
        or config.continuous_pca_post_action_s <= 0
        or config.continuous_pca_sample_step_s <= 0
    ):
        raise ValueError("Continuous-PCA time settings must be positive.")
    for window_name, window_s in (
        ("continuous_pca_pre_stim_s", config.continuous_pca_pre_stim_s),
        (
            "continuous_pca_post_action_s",
            config.continuous_pca_post_action_s,
        ),
    ):
        n_steps = window_s / config.continuous_pca_sample_step_s
        if not np.isclose(n_steps, round(n_steps)):
            raise ValueError(
                f"{window_name} must be an integer multiple of "
                "continuous_pca_sample_step_s."
            )
    if config.continuous_pca_decision_points < 3:
        raise ValueError(
            "continuous_pca_decision_points must be at least 3."
        )
    if config.continuous_pca_smoothing_s <= 0:
        raise ValueError("continuous_pca_smoothing_s must be positive.")
    if config.continuous_pca_max_individual_trials_per_group < 1:
        raise ValueError(
            "continuous_pca_max_individual_trials_per_group must be positive."
        )
    if not 0 <= config.continuous_pca_individual_alpha <= 1:
        raise ValueError(
            "continuous_pca_individual_alpha must be between 0 and 1."
        )
    if not config.expected_priors:
        raise ValueError("expected_priors cannot be empty.")
    if not config.contrast_levels:
        raise ValueError("contrast_levels cannot be empty.")
    if config.min_trials_per_peth_condition < 1:
        raise ValueError("min_trials_per_peth_condition must be positive.")
    if config.min_good_units_per_region < 1:
        raise ValueError("min_good_units_per_region must be positive.")
    supported_formats = {"png"}
    if not config.figure_formats:
        raise ValueError("figure_formats cannot be empty.")
    if any(fmt.lower() not in supported_formats for fmt in config.figure_formats):
        raise ValueError(
            f"figure_formats must be drawn from {sorted(supported_formats)}."
        )
    if config.figure_dpi < 150:
        raise ValueError("figure_dpi must be at least 150.")
    if not config.figure_output_dir.strip():
        raise ValueError("figure_output_dir cannot be empty.")


def map_requested_regions(
    brain_regions: BrainRegions,
    regions: Iterable[str],
    mapping: str,
) -> tuple[str, ...]:
    requested = np.asarray(list(regions), dtype=str)
    mapped = np.asarray(
        brain_regions.acronym2acronym(requested, mapping=mapping), dtype=str
    )
    invalid = requested[np.isin(mapped, ["root", "void", "nan", "None", ""])]
    if len(invalid):
        raise ValueError(
            f"Regions could not be mapped with {mapping}: {invalid.tolist()}"
        )
    if len(set(mapped.tolist())) != len(mapped):
        raise ValueError(
            "Requested regions are not unique after atlas mapping: "
            f"{dict(zip(requested.tolist(), mapped.tolist()))}"
        )
    return tuple(mapped.tolist())


def make_error(stage: str, eid: str, region: str | None, exc: Exception) -> dict:
    return {
        "stage": stage,
        "eid": eid,
        "region": region,
        "error_type": type(exc).__name__,
        "message": str(exc),
    }


## 3. Trial and hesitation logic

Hesitation is the official wheel-velocity direction-change count from stimulus onset
to response time. No additional stability threshold is applied. A score of zero is kept
as the explicit `No hesitation` state but is excluded from Q1–Q4. Positive scores are
ranked independently within each session. Seeded tie-breaking keeps four reproducible,
nearly equal-count positive-score groups when a discrete score crosses a boundary; the
QC figure shows the exact assignment.


In [ ]:
def load_trials(one: ONE, eid: str) -> dict[str, np.ndarray]:
    trials = one.load_object(eid, "trials", collection="alf")
    missing = REQUIRED_TRIAL_FIELDS - set(trials.keys())
    if missing:
        raise ValueError(f"Missing trial fields: {sorted(missing)}")
    return {
        name: np.asarray(trials[name], dtype=float)
        for name in REQUIRED_TRIAL_FIELDS
    }


def prepare_trial_context(trials: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    left_contrast = trials["contrastLeft"]
    right_contrast = trials["contrastRight"]

    left_stimulus = np.isfinite(left_contrast) & np.isnan(right_contrast)
    right_stimulus = np.isnan(left_contrast) & np.isfinite(right_contrast)
    unambiguous_side = left_stimulus | right_stimulus

    peth_contrast = np.where(
        left_stimulus,
        left_contrast,
        right_contrast,
    )
    decoder_contrast = np.where(
        np.isfinite(left_contrast),
        left_contrast,
        right_contrast,
    )
    movement_latency = (
        trials["firstMovement_times"] - trials["stimOn_times"]
    )
    return {
        **trials,
        "left_stimulus": left_stimulus,
        "right_stimulus": right_stimulus,
        "unambiguous_side": unambiguous_side,
        "peth_contrast": peth_contrast,
        "decoder_contrast": decoder_contrast,
        "movement_latency": movement_latency,
    }


def compute_hesitation_scores(
    wheel_times: np.ndarray,
    wheel_velocity: np.ndarray,
    movement_intervals: np.ndarray,
    trial_starts: np.ndarray,
    trial_ends: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    '''Count official wheel direction changes in each trial score window.'''
    wheel_times = np.asarray(wheel_times, dtype=float)
    wheel_velocity = np.asarray(wheel_velocity, dtype=float)
    trial_starts = np.asarray(trial_starts, dtype=float)
    trial_ends = np.asarray(trial_ends, dtype=float)
    intervals = np.asarray(movement_intervals, dtype=float)

    if wheel_times.ndim != 1 or wheel_velocity.ndim != 1:
        raise ValueError("Wheel times and velocity must be one-dimensional.")
    if len(wheel_times) != len(wheel_velocity) or len(wheel_times) < 2:
        raise ValueError("Wheel times and velocity must have equal non-trivial length.")
    if not np.all(np.isfinite(wheel_times)) or not np.all(
        np.isfinite(wheel_velocity)
    ):
        raise ValueError("Wheel times and velocity must be finite.")
    if not np.all(np.diff(wheel_times) > 0):
        raise ValueError("Wheel times must be strictly increasing.")
    if trial_starts.shape != trial_ends.shape:
        raise ValueError("Trial score-window starts and ends must have equal shape.")
    if intervals.size == 0:
        intervals = np.empty((0, 2), dtype=float)
    else:
        intervals = intervals.reshape(-1, 2)
        intervals = intervals[
            np.all(np.isfinite(intervals), axis=1)
            & (intervals[:, 1] > intervals[:, 0])
        ]

    changes_by_movement, _ = direction_changes(
        wheel_times,
        wheel_velocity,
        intervals,
    )
    nonempty = [
        np.asarray(values, dtype=float)
        for values in changes_by_movement
        if len(values)
    ]
    change_times = (
        np.unique(np.concatenate(nonempty))
        if nonempty
        else np.asarray([], dtype=float)
    )

    scores = np.full(trial_starts.shape, np.nan, dtype=float)
    covered = (
        np.isfinite(trial_starts)
        & np.isfinite(trial_ends)
        & (trial_ends > trial_starts)
        & (trial_starts >= wheel_times[0])
        & (trial_ends <= wheel_times[-1])
    )
    for trial_id in np.flatnonzero(covered):
        left = np.searchsorted(
            change_times, trial_starts[trial_id], side="left"
        )
        right = np.searchsorted(
            change_times, trial_ends[trial_id], side="right"
        )
        scores[trial_id] = float(right - left)
    return scores, change_times


def assign_hesitation_quartiles(
    scores: np.ndarray,
    n_quartiles: int,
    random_state: int,
) -> np.ndarray:
    '''Create reproducible rank quartiles from positive scores only.'''
    scores = np.asarray(scores, dtype=float)
    if n_quartiles < 2:
        raise ValueError("n_quartiles must be at least 2.")
    finite = np.isfinite(scores)
    if np.any(scores[finite] < 0):
        raise ValueError("Hesitation scores cannot be negative.")
    positive_ids = np.flatnonzero(finite & (scores >= 1.0))
    if len(positive_ids) < n_quartiles:
        raise ValueError(
            f"Only {len(positive_ids)} positive hesitation scores for "
            f"{n_quartiles} quartiles; score-zero trials are not quartiled."
        )

    # Direction-change scores are discrete. A seeded random key breaks a
    # boundary tie without trial-order bias. Score remains the primary
    # ordering key and positive-score group sizes differ by at most one.
    rng = np.random.default_rng(random_state)
    tie_breaker = rng.random(len(positive_ids))
    order = np.lexsort((tie_breaker, scores[positive_ids]))
    ranks = np.empty(len(positive_ids), dtype=int)
    ranks[order] = np.arange(len(positive_ids), dtype=int)
    labels = np.minimum(
        ranks * n_quartiles // len(positive_ids) + 1,
        n_quartiles,
    )

    quartiles = np.full(scores.shape, np.nan, dtype=float)
    quartiles[positive_ids] = labels
    return quartiles


def build_hesitation_result(
    context: dict[str, np.ndarray],
    scores: np.ndarray,
    direction_change_times: np.ndarray,
    config: ProcessingConfig,
) -> HesitationResult:
    quartiles = assign_hesitation_quartiles(
        scores,
        config.hesitation_n_quartiles,
        config.random_state,
    )
    has_hesitation = np.full(np.asarray(scores).shape, np.nan, dtype=float)
    finite_scores = np.isfinite(scores)
    has_hesitation[finite_scores] = (
        np.asarray(scores, dtype=float)[finite_scores] >= 1.0
    ).astype(float)
    no_hesitation_vs_q4 = np.full(
        np.asarray(scores).shape, np.nan, dtype=float
    )
    no_hesitation_vs_q4[has_hesitation == 0.0] = 0.0
    no_hesitation_vs_q4[
        np.isclose(
            quartiles,
            float(config.hesitation_n_quartiles),
            equal_nan=False,
        )
    ] = float(config.hesitation_n_quartiles)
    trial_table = pd.DataFrame(
        {
            "trial_id": np.arange(len(scores), dtype=int),
            "stim_on_time": context["stimOn_times"],
            "response_time": context["response_times"],
            "hesitation_score": scores,
            "has_hesitation": has_hesitation,
            "hesitation_quartile": quartiles,
            "no_hesitation_vs_q4": no_hesitation_vs_q4,
            "contrast": context["decoder_contrast"],
            "choice": context["choice"],
        }
    )
    valid_table = trial_table.dropna(
        subset=["hesitation_score", "hesitation_quartile"]
    ).copy()
    valid_table["hesitation_quartile"] = valid_table[
        "hesitation_quartile"
    ].astype(int)
    summary = (
        valid_table.groupby("hesitation_quartile", as_index=False)
        .agg(
            n_trials=("trial_id", "size"),
            score_min=("hesitation_score", "min"),
            score_median=("hesitation_score", "median"),
            score_max=("hesitation_score", "max"),
        )
        .set_index("hesitation_quartile")
        .reindex(range(1, config.hesitation_n_quartiles + 1))
        .reset_index()
    )
    summary.insert(1, "label", [
        f"Q{value}" for value in summary["hesitation_quartile"]
    ])
    positive_total = int(summary["n_trials"].sum())
    summary["percent_hesitation_trials"] = (
        100.0 * summary["n_trials"] / positive_total
    )

    state_table = trial_table.dropna(
        subset=["hesitation_score", "has_hesitation"]
    ).copy()
    state_table["has_hesitation"] = state_table[
        "has_hesitation"
    ].astype(int)
    state_summary = (
        state_table.groupby("has_hesitation", as_index=False)
        .agg(
            n_trials=("trial_id", "size"),
            score_min=("hesitation_score", "min"),
            score_median=("hesitation_score", "median"),
            score_max=("hesitation_score", "max"),
        )
        .set_index("has_hesitation")
        .reindex([0, 1])
        .reset_index()
    )
    state_summary.insert(
        1,
        "label",
        state_summary["has_hesitation"].map(
            {0: "No hesitation", 1: "Hesitation"}
        ),
    )
    state_summary["percent_trials"] = (
        100.0 * state_summary["n_trials"] / len(state_table)
    )
    return HesitationResult(
        scores=np.asarray(scores, dtype=float),
        quartiles=quartiles,
        has_hesitation=has_hesitation,
        no_hesitation_vs_q4=no_hesitation_vs_q4,
        direction_change_times=np.asarray(
            direction_change_times, dtype=float
        ),
        trial_table=trial_table,
        summary=summary,
        state_summary=state_summary,
    )


def load_hesitation_result(
    one: ONE,
    eid: str,
    context: dict[str, np.ndarray],
    config: ProcessingConfig,
) -> HesitationResult:
    loader = SessionLoader(one=one, eid=eid)
    loader.load_wheel(
        fs=config.wheel_sampling_hz,
        corner_frequency=config.wheel_corner_frequency_hz,
        order=config.wheel_filter_order,
    )
    wheel_moves = one.load_object(
        eid, "wheelMoves", collection="alf"
    )
    if "intervals" not in wheel_moves:
        raise ValueError("wheelMoves.intervals is unavailable.")

    scores, change_times = compute_hesitation_scores(
        loader.wheel["times"].to_numpy(dtype=float),
        loader.wheel["velocity"].to_numpy(dtype=float),
        np.asarray(wheel_moves["intervals"], dtype=float),
        context["stimOn_times"],
        context["response_times"],
    )
    return build_hesitation_result(
        context,
        scores,
        change_times,
        config,
    )


def attach_hesitation_to_context(
    context: dict[str, np.ndarray],
    hesitation: HesitationResult,
) -> dict[str, np.ndarray]:
    if len(hesitation.scores) != len(context["choice"]):
        raise ValueError("Hesitation scores and trials have different lengths.")
    return {
        **context,
        "hesitation_score": hesitation.scores,
        "has_hesitation": hesitation.has_hesitation,
        "hesitation_quartile": hesitation.quartiles,
        "no_hesitation_vs_q4": (
            hesitation.no_hesitation_vs_q4
        ),
    }


def contralateral_trial_mask(context: dict[str, np.ndarray], hemisphere: str) -> np.ndarray:
    if hemisphere == "left":
        return context["right_stimulus"]
    if hemisphere == "right":
        return context["left_stimulus"]
    raise ValueError(f"Hemisphere must be left or right, received {hemisphere!r}.")


## 4. Select one probe per region and load spikes

Cluster IDs are remapped before spikes from different selected probes are combined.


In [ ]:
def _session_subject(one: ONE, eid: str) -> str:
    record = one.alyx.rest("sessions", "read", id=eid)
    subject = record.get("subject", "unknown")
    if isinstance(subject, dict):
        subject = subject.get("nickname", "unknown")
    return str(subject)


def _strict_hemisphere(x: np.ndarray) -> str | None:
    values = np.asarray(x, dtype=float)
    if not len(values):
        return None
    if np.all(values < 0):
        return "left"
    if np.all(values > 0):
        return "right"
    return "mixed"


def _load_probe_region_metadata(
    one: ONE,
    eid: str,
    probe_name: str,
    config: ProcessingConfig,
    brain_regions: BrainRegions,
) -> tuple[pd.DataFrame, str]:
    loader = SpikeSortingLoader(eid=eid, pname=probe_name, one=one)
    clusters = loader.load_spike_sorting_object("clusters")
    channels = loader.load_channels()

    if "metrics" not in clusters:
        raise ValueError(f"{probe_name}: clusters.metrics not found.")
    metrics = pd.DataFrame(clusters["metrics"])
    if "label" not in metrics:
        raise ValueError(f"{probe_name}: cluster QC label not found.")

    cluster_channels = np.asarray(clusters["channels"], dtype=int)
    if len(metrics) != len(cluster_channels):
        raise ValueError(
            f"{probe_name}: cluster metrics and channels have different lengths."
        )

    channel_acronyms = np.asarray(channels["acronym"])
    channel_x = np.asarray(channels["x"], dtype=float)
    allen = channel_acronyms[cluster_channels]
    x = channel_x[cluster_channels]
    mapped = brain_regions.acronym2acronym(
        allen, mapping=config.atlas_mapping
    )

    metadata = pd.DataFrame(
        {
            "is_good": metrics["label"].to_numpy() == 1,
            "region": np.asarray(mapped, dtype=str),
            "x": x,
        }
    )
    return metadata, loader.histology


def select_region_probes(
    one: ONE,
    eid: str,
    mapped_regions: tuple[str, ...],
    config: ProcessingConfig,
    brain_regions: BrainRegions,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    insertions = one.alyx.rest("insertions", "list", session=eid)
    if not insertions:
        raise ValueError("No probe insertions found for this session.")

    candidate_rows: list[dict] = []
    error_rows: list[dict] = []

    for insertion in insertions:
        probe_name = insertion["name"]
        pid = str(insertion.get("id", ""))
        if not pid:
            raise ValueError(f"{probe_name}: insertion PID is missing.")

        try:
            metadata, histology = _load_probe_region_metadata(
                one, eid, probe_name, config, brain_regions
            )
            if histology not in config.accepted_histology:
                continue

            for requested, region in zip(config.regions, mapped_regions):
                units = metadata[
                    metadata["is_good"] & metadata["region"].eq(region)
                ]
                candidate_rows.append(
                    {
                        "eid": eid,
                        "requested_region": requested,
                        "region": region,
                        "pid": pid,
                        "probe_name": probe_name,
                        "histology": histology,
                        "n_good_units": len(units),
                        "hemisphere": _strict_hemisphere(
                            units["x"].to_numpy(dtype=float)
                        ),
                    }
                )
        except Exception as exc:
            error_rows.append(make_error("probe_metadata", eid, None, exc))
            if not config.continue_on_error:
                raise

    candidates = pd.DataFrame(
        candidate_rows,
        columns=[
            "eid",
            "requested_region",
            "region",
            "pid",
            "probe_name",
            "histology",
            "n_good_units",
            "hemisphere",
        ],
    )
    selected_rows: list[dict] = []
    for requested, region in zip(config.regions, mapped_regions):
        region_candidates = (
            candidates[candidates["region"].eq(region)]
            .sort_values(
                ["n_good_units", "probe_name", "pid"],
                ascending=[False, True, True],
            )
            .reset_index(drop=True)
        )
        if region_candidates.empty:
            raise ValueError(f"{region}: no accepted-histology probe found.")

        selected = region_candidates.iloc[0].to_dict()
        if selected["n_good_units"] < config.min_good_units_per_region:
            raise ValueError(
                f"{region}: {selected['n_good_units']} good units; "
                f"required {config.min_good_units_per_region}."
            )
        if selected["hemisphere"] not in {"left", "right"}:
            raise ValueError(
                f"{region}: good units on {selected['probe_name']} are not "
                "confined to one hemisphere."
            )
        selected["requested_region"] = requested
        selected_rows.append(selected)

    selected_probes = pd.DataFrame(selected_rows)
    selected_probes["recording_start"] = np.nan
    selected_probes["recording_end"] = np.nan
    return selected_probes, pd.DataFrame(
        error_rows,
        columns=["stage", "eid", "region", "error_type", "message"],
    )


def load_neural_session(
    one: ONE,
    eid: str,
    mapped_regions: tuple[str, ...],
    config: ProcessingConfig,
    atlas: AllenAtlas,
    brain_regions: BrainRegions,
) -> NeuralSessionData:
    subject = _session_subject(one, eid)
    region_probes, probe_errors = select_region_probes(
        one, eid, mapped_regions, config, brain_regions
    )

    selected_time_parts: list[np.ndarray] = []
    selected_unit_parts: list[np.ndarray] = []
    unit_rows: list[dict] = []
    next_global_id = 0

    for (pid, probe_name), selected_regions in region_probes.groupby(
        ["pid", "probe_name"], sort=False
    ):
        try:
            loader = SpikeSortingLoader(pid=pid, one=one, atlas=atlas)
            spikes, clusters, channels = loader.load_spike_sorting()
            clusters = loader.merge_clusters(spikes, clusters, channels)

            if loader.histology not in config.accepted_histology:
                raise ValueError(
                    f"{probe_name}: histology {loader.histology!r} is not accepted."
                )
            required_cluster_fields = {"cluster_id", "label", "acronym", "x"}
            missing = required_cluster_fields - set(clusters.keys())
            if missing:
                raise ValueError(
                    f"{probe_name}: missing merged cluster fields {sorted(missing)}."
                )

            spike_times = np.asarray(spikes["times"], dtype=float)
            spike_clusters = np.asarray(spikes["clusters"], dtype=int)
            if not len(spike_times):
                raise ValueError(f"{probe_name}: no spikes found.")

            mapped = np.asarray(
                brain_regions.acronym2acronym(
                    clusters["acronym"], mapping=config.atlas_mapping
                ),
                dtype=str,
            )
            cluster_ids = np.asarray(clusters["cluster_id"], dtype=int)
            quality = np.asarray(clusters["label"]) == 1
            cluster_x = np.asarray(clusters["x"], dtype=float)
            probe_local_ids: list[np.ndarray] = []
            probe_global_ids: list[np.ndarray] = []

            recording_start = float(np.nanmin(spike_times))
            recording_end = float(np.nanmax(spike_times))

            for region_row in selected_regions.itertuples(index=False):
                selected = quality & (mapped == region_row.region)
                local_ids = cluster_ids[selected]
                selected_x = cluster_x[selected]
                hemisphere = _strict_hemisphere(selected_x)

                if len(local_ids) < config.min_good_units_per_region:
                    raise ValueError(
                        f"{region_row.region}: {len(local_ids)} good units after "
                        "loading merged clusters."
                    )
                if hemisphere not in {"left", "right"}:
                    raise ValueError(
                        f"{region_row.region}: units are not confined to one hemisphere."
                    )

                global_ids = np.arange(
                    next_global_id,
                    next_global_id + len(local_ids),
                    dtype=int,
                )
                next_global_id += len(local_ids)
                probe_local_ids.append(local_ids)
                probe_global_ids.append(global_ids)

                for local_id, global_id, x_value in zip(
                    local_ids, global_ids, selected_x
                ):
                    unit_rows.append(
                        {
                            "eid": eid,
                            "subject": subject,
                            "pid": pid,
                            "probe_name": probe_name,
                            "cluster_id": int(local_id),
                            "unit_id": int(global_id),
                            "region": region_row.region,
                            "x": float(x_value),
                            "hemisphere": hemisphere,
                        }
                    )

                row_mask = (
                    region_probes["pid"].eq(pid)
                    & region_probes["probe_name"].eq(probe_name)
                    & region_probes["region"].eq(region_row.region)
                )
                region_probes.loc[row_mask, "n_good_units"] = len(local_ids)
                region_probes.loc[row_mask, "hemisphere"] = hemisphere
                region_probes.loc[row_mask, "recording_start"] = recording_start
                region_probes.loc[row_mask, "recording_end"] = recording_end

            local_ids = np.concatenate(probe_local_ids)
            global_ids = np.concatenate(probe_global_ids)
            order = np.argsort(local_ids)
            sorted_local_ids = local_ids[order]
            sorted_global_ids = global_ids[order]
            keep_spikes = np.isin(spike_clusters, sorted_local_ids)
            kept_local = spike_clusters[keep_spikes]
            positions = np.searchsorted(sorted_local_ids, kept_local)
            selected_time_parts.append(spike_times[keep_spikes])
            selected_unit_parts.append(sorted_global_ids[positions])

            del spikes, clusters, channels, spike_times, spike_clusters
            gc.collect()
        except Exception as exc:
            new_error = pd.DataFrame(
                [
                    make_error(
                        "load_probe",
                        eid,
                        None,
                        RuntimeError(f"{probe_name}: {exc}"),
                    )
                ]
            )
            probe_errors = pd.concat(
                [probe_errors, new_error], ignore_index=True
            )
            if not config.continue_on_error:
                raise

    if not unit_rows:
        raise ValueError(
            f"No good units found in requested regions {list(mapped_regions)}."
        )

    units = pd.DataFrame(unit_rows)
    spike_times = np.concatenate(selected_time_parts)
    spike_units = np.concatenate(selected_unit_parts)
    if not len(spike_times):
        raise ValueError("Selected units contain no spikes.")
    chronological = np.argsort(spike_times, kind="stable")
    spike_times = spike_times[chronological]
    spike_units = spike_units[chronological]

    return NeuralSessionData(
        eid=eid,
        subject=subject,
        spike_times=spike_times,
        spike_units=spike_units,
        units=units,
        region_probes=region_probes.reset_index(drop=True),
        probe_errors=probe_errors,
    )


## 5. Prior-balanced population PETHs

Contrast PETHs require every configured prior. For hesitation-grouped PETHs, balancing
uses the same prior subset for every compared group: only priors meeting
`min_trials_per_peth_condition` in all groups are retained. The condition tables mark
retained priors as `used`; if none remain, only that grouped PETH is skipped.


In [ ]:
def peth_base_mask(
    context: dict[str, np.ndarray],
    neural: NeuralSessionData,
    region: str,
    config: ProcessingConfig,
) -> np.ndarray:
    low, high = config.movement_latency_s
    stim = context["stimOn_times"]
    action = context["firstMovement_times"]
    latency = context["movement_latency"]
    probe = neural.region_probes[
        neural.region_probes["region"].eq(region)
    ]
    if len(probe) != 1:
        raise ValueError(f"{region}: expected one selected probe, found {len(probe)}.")
    recording_start = float(probe.iloc[0]["recording_start"])
    recording_end = float(probe.iloc[0]["recording_end"])

    return (
        np.isfinite(context["peth_contrast"])
        & np.isfinite(context["probabilityLeft"])
        & np.isfinite(context["choice"])
        & np.isfinite(stim)
        & np.isfinite(action)
        & (latency >= low)
        & (latency <= high)
        & (stim - config.peth_pre_s >= recording_start)
        & (stim + config.peth_post_s <= recording_end)
        & (action - config.peth_pre_s >= recording_start)
        & (action + config.peth_post_s <= recording_end)
    )


def _peth_means(peths: Any) -> np.ndarray:
    if "means" in peths:
        return np.asarray(peths["means"], dtype=float)
    if "mean" in peths:
        return np.asarray(peths["mean"], dtype=float)
    raise KeyError("calculate_peths output has neither 'means' nor 'mean'.")


def _compute_prior_balanced_population_peth(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    region: str,
    event_times: np.ndarray,
    base_mask: np.ndarray,
    condition_values: np.ndarray,
    condition_level: float,
    condition_name: str,
    config: ProcessingConfig,
    priors: tuple[float, ...] | None = None,
) -> PopulationPETH:
    region_units = neural.units[neural.units["region"].eq(region)]
    hemispheres = region_units["hemisphere"].unique().tolist()
    if len(hemispheres) != 1 or hemispheres[0] not in {"left", "right"}:
        raise ValueError(f"{region}: expected one left/right hemisphere.")

    hemisphere = hemispheres[0]
    cluster_ids = region_units["unit_id"].to_numpy(dtype=int)
    contralateral = contralateral_trial_mask(context, hemisphere)
    prior_parts: list[np.ndarray] = []
    time_scale: np.ndarray | None = None

    selected_priors = (
        config.expected_priors if priors is None else tuple(priors)
    )
    if not selected_priors:
        raise ValueError(f"{region}: no priors are available for PETH balancing.")

    for prior in selected_priors:
        trial_mask = (
            base_mask
            & contralateral
            & np.isclose(condition_values, condition_level)
            & np.isclose(context["probabilityLeft"], prior)
        )
        n_trials = int(trial_mask.sum())
        if n_trials < config.min_trials_per_peth_condition:
            raise ValueError(
                f"{region}: {condition_name}={condition_level}, "
                f"prior={prior} has "
                f"{n_trials} trials; required "
                f"{config.min_trials_per_peth_condition}."
            )

        peths, _ = calculate_peths(
            spike_times=neural.spike_times,
            spike_clusters=neural.spike_units,
            cluster_ids=cluster_ids,
            align_times=event_times[trial_mask],
            pre_time=config.peth_pre_s,
            post_time=config.peth_post_s,
            bin_size=config.peth_bin_s,
            smoothing=config.peth_smoothing_s,
            return_fr=True,
        )
        prior_parts.append(_peth_means(peths))
        current_time = np.asarray(peths["tscale"], dtype=float)
        if time_scale is None:
            time_scale = current_time
        elif not np.allclose(time_scale, current_time):
            raise ValueError("Inconsistent PETH time scales across conditions.")

    if time_scale is None:
        raise ValueError(f"{region}: PETH time scale was not produced.")

    balanced_units = np.mean(np.stack(prior_parts, axis=0), axis=0)
    mean = np.mean(balanced_units, axis=0)
    sem = np.std(balanced_units, axis=0, ddof=1) / np.sqrt(
        balanced_units.shape[0]
    )
    return PopulationPETH(
        time_s=time_scale,
        mean_hz=mean,
        sem_hz=sem,
        n_units=balanced_units.shape[0],
    )


def compute_region_population_peth(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    region: str,
    event_times: np.ndarray,
    base_mask: np.ndarray,
    contrast_level: float,
    config: ProcessingConfig,
) -> PopulationPETH:
    return _compute_prior_balanced_population_peth(
        neural,
        context,
        region,
        event_times,
        base_mask,
        context["peth_contrast"],
        contrast_level,
        "contrast",
        config,
    )


def grouped_peth_base_mask(
    context: dict[str, np.ndarray],
    neural: NeuralSessionData,
    region: str,
    group_name: str,
    config: ProcessingConfig,
) -> np.ndarray:
    if group_name not in context:
        raise KeyError(f"Trial context is missing {group_name!r}.")
    group_values = np.asarray(context[group_name], dtype=float)
    return peth_base_mask(
        context, neural, region, config
    ) & np.isfinite(group_values)


def compute_region_grouped_population_peth(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    region: str,
    event_times: np.ndarray,
    base_mask: np.ndarray,
    group_name: str,
    group_level: int,
    config: ProcessingConfig,
    priors: tuple[float, ...] | None = None,
) -> PopulationPETH:
    return _compute_prior_balanced_population_peth(
        neural,
        context,
        region,
        event_times,
        base_mask,
        np.asarray(context[group_name], dtype=float),
        group_level,
        group_name,
        config,
        priors,
    )


def hesitation_peth_base_mask(
    context: dict[str, np.ndarray],
    neural: NeuralSessionData,
    region: str,
    config: ProcessingConfig,
) -> np.ndarray:
    return grouped_peth_base_mask(
        context, neural, region, "hesitation_quartile", config
    )


def hesitation_presence_peth_base_mask(
    context: dict[str, np.ndarray],
    neural: NeuralSessionData,
    region: str,
    config: ProcessingConfig,
) -> np.ndarray:
    return grouped_peth_base_mask(
        context, neural, region, "has_hesitation", config
    )


def no_hesitation_vs_q4_peth_base_mask(
    context: dict[str, np.ndarray],
    neural: NeuralSessionData,
    region: str,
    config: ProcessingConfig,
) -> np.ndarray:
    return grouped_peth_base_mask(
        context, neural, region, "no_hesitation_vs_q4", config
    )


def compute_region_hesitation_population_peth(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    region: str,
    event_times: np.ndarray,
    base_mask: np.ndarray,
    quartile: int,
    config: ProcessingConfig,
    priors: tuple[float, ...] | None = None,
) -> PopulationPETH:
    return compute_region_grouped_population_peth(
        neural,
        context,
        region,
        event_times,
        base_mask,
        "hesitation_quartile",
        quartile,
        config,
        priors,
    )


def compute_region_hesitation_presence_population_peth(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    region: str,
    event_times: np.ndarray,
    base_mask: np.ndarray,
    has_hesitation: int,
    config: ProcessingConfig,
    priors: tuple[float, ...] | None = None,
) -> PopulationPETH:
    return compute_region_grouped_population_peth(
        neural,
        context,
        region,
        event_times,
        base_mask,
        "has_hesitation",
        has_hesitation,
        config,
        priors,
    )


def compute_region_no_hesitation_vs_q4_population_peth(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    region: str,
    event_times: np.ndarray,
    base_mask: np.ndarray,
    group_level: int,
    config: ProcessingConfig,
    priors: tuple[float, ...] | None = None,
) -> PopulationPETH:
    return compute_region_grouped_population_peth(
        neural,
        context,
        region,
        event_times,
        base_mask,
        "no_hesitation_vs_q4",
        group_level,
        config,
        priors,
    )


def peth_condition_table(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    mapped_regions: tuple[str, ...],
    config: ProcessingConfig,
) -> pd.DataFrame:
    rows: list[dict] = []
    for region in mapped_regions:
        region_units = neural.units[neural.units["region"].eq(region)]
        hemispheres = region_units["hemisphere"].unique().tolist()
        if len(hemispheres) != 1 or hemispheres[0] not in {"left", "right"}:
            raise ValueError(f"{region}: expected one left/right hemisphere.")
        hemisphere = hemispheres[0]
        n_units = len(region_units)
        contra = contralateral_trial_mask(context, hemisphere)
        region_base_mask = peth_base_mask(context, neural, region, config)
        for contrast in config.contrast_levels:
            for prior in config.expected_priors:
                mask = (
                    region_base_mask
                    & contra
                    & np.isclose(context["peth_contrast"], contrast)
                    & np.isclose(context["probabilityLeft"], prior)
                )
                rows.append(
                    {
                        "eid": neural.eid,
                        "region": region,
                        "hemisphere": hemisphere,
                        "n_units": n_units,
                        "contrast": contrast,
                        "probability_left": prior,
                        "n_trials": int(mask.sum()),
                    }
                )
    return pd.DataFrame(
        rows,
        columns=[
            "eid",
            "region",
            "hemisphere",
            "n_units",
            "contrast",
            "probability_left",
            "n_trials",
        ],
    )


def grouped_peth_condition_table(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    mapped_regions: tuple[str, ...],
    group_name: str,
    group_levels: tuple[int, ...],
    config: ProcessingConfig,
) -> pd.DataFrame:
    rows: list[dict] = []
    for region in mapped_regions:
        region_units = neural.units[neural.units["region"].eq(region)]
        hemispheres = region_units["hemisphere"].unique().tolist()
        if len(hemispheres) != 1 or hemispheres[0] not in {"left", "right"}:
            raise ValueError(f"{region}: expected one left/right hemisphere.")
        hemisphere = hemispheres[0]
        contra = contralateral_trial_mask(context, hemisphere)
        base_mask = grouped_peth_base_mask(
            context, neural, region, group_name, config
        )
        for group_level in group_levels:
            for prior in config.expected_priors:
                mask = (
                    base_mask
                    & contra
                    & np.isclose(
                        context[group_name], group_level
                    )
                    & np.isclose(context["probabilityLeft"], prior)
                )
                rows.append(
                    {
                        "eid": neural.eid,
                        "region": region,
                        "hemisphere": hemisphere,
                        "n_units": len(region_units),
                        group_name: group_level,
                        "probability_left": prior,
                        "n_trials": int(mask.sum()),
                    }
                )
    counts = pd.DataFrame(rows)
    counts["used_for_balancing"] = False
    counts["prior_status"] = "not_evaluated"
    return counts


def shared_grouped_peth_priors(
    counts: pd.DataFrame,
    region: str,
    group_name: str,
    group_levels: tuple[int, ...],
    config: ProcessingConfig,
) -> tuple[float, ...]:
    '''Return priors with enough trials in every requested group.'''
    region_counts = counts.loc[counts["region"].eq(region)]
    shared: list[float] = []
    for prior in config.expected_priors:
        prior_counts = region_counts.loc[
            np.isclose(region_counts["probability_left"], prior)
        ]
        level_counts = {
            int(level): int(
                prior_counts.loc[
                    np.isclose(prior_counts[group_name], level),
                    "n_trials",
                ].sum()
            )
            for level in group_levels
        }
        if all(
            level_counts[int(level)]
            >= config.min_trials_per_peth_condition
            for level in group_levels
        ):
            shared.append(float(prior))
    return tuple(shared)


def hesitation_peth_condition_table(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    mapped_regions: tuple[str, ...],
    config: ProcessingConfig,
) -> pd.DataFrame:
    return grouped_peth_condition_table(
        neural,
        context,
        mapped_regions,
        "hesitation_quartile",
        tuple(range(1, config.hesitation_n_quartiles + 1)),
        config,
    )


def hesitation_presence_peth_condition_table(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    mapped_regions: tuple[str, ...],
    config: ProcessingConfig,
) -> pd.DataFrame:
    return grouped_peth_condition_table(
        neural,
        context,
        mapped_regions,
        "has_hesitation",
        (0, 1),
        config,
    )


def no_hesitation_vs_q4_peth_condition_table(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    mapped_regions: tuple[str, ...],
    config: ProcessingConfig,
) -> pd.DataFrame:
    return grouped_peth_condition_table(
        neural,
        context,
        mapped_regions,
        "no_hesitation_vs_q4",
        (0, config.hesitation_n_quartiles),
        config,
    )


def run_peth_analysis(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    mapped_regions: tuple[str, ...],
    config: ProcessingConfig,
) -> tuple[
    dict[str, dict[str, dict[float, PopulationPETH]]],
    pd.DataFrame,
    pd.DataFrame,
]:
    counts = peth_condition_table(
        neural, context, mapped_regions, config
    )
    results: dict[str, dict[str, dict[float, PopulationPETH]]] = {}
    errors: list[dict] = []

    for region in mapped_regions:
        try:
            base_mask = peth_base_mask(context, neural, region, config)
            results[region] = {"stimulus": {}, "action": {}}
            for contrast in config.contrast_levels:
                results[region]["stimulus"][contrast] = (
                    compute_region_population_peth(
                        neural,
                        context,
                        region,
                        context["stimOn_times"],
                        base_mask,
                        contrast,
                        config,
                    )
                )
                results[region]["action"][contrast] = (
                    compute_region_population_peth(
                        neural,
                        context,
                        region,
                        context["firstMovement_times"],
                        base_mask,
                        contrast,
                        config,
                    )
                )
        except Exception as exc:
            results.pop(region, None)
            errors.append(make_error("peth", neural.eid, region, exc))
            if not config.continue_on_error:
                raise

    return results, counts, pd.DataFrame(errors)


def run_grouped_peth_analysis(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    mapped_regions: tuple[str, ...],
    group_name: str,
    group_levels: tuple[int, ...],
    error_stage: str,
    config: ProcessingConfig,
) -> tuple[
    dict[str, dict[str, dict[int, PopulationPETH]]],
    pd.DataFrame,
    pd.DataFrame,
]:
    counts = grouped_peth_condition_table(
        neural,
        context,
        mapped_regions,
        group_name,
        group_levels,
        config,
    )
    results: dict[
        str, dict[str, dict[int, PopulationPETH]]
    ] = {}
    errors: list[dict] = []

    for region in mapped_regions:
        shared_priors = shared_grouped_peth_priors(
            counts,
            region,
            group_name,
            group_levels,
            config,
        )
        region_rows = counts["region"].eq(region)
        used_rows = region_rows & counts["probability_left"].apply(
            lambda prior: any(
                np.isclose(prior, selected)
                for selected in shared_priors
            )
        )
        counts.loc[region_rows, "prior_status"] = "excluded_sparse"
        counts.loc[used_rows, "used_for_balancing"] = True
        counts.loc[used_rows, "prior_status"] = "used"

        if not shared_priors:
            error = ValueError(
                f"{region}: no probabilityLeft prior has at least "
                f"{config.min_trials_per_peth_condition} trial(s) "
                f"in every {group_name} group. This grouped PETH was "
                "skipped; independent analyses will continue."
            )
            errors.append(
                make_error(error_stage, neural.eid, region, error)
            )
            continue

        try:
            base_mask = grouped_peth_base_mask(
                context, neural, region, group_name, config
            )
            results[region] = {"stimulus": {}, "action": {}}
            for group_level in group_levels:
                results[region]["stimulus"][group_level] = (
                    compute_region_grouped_population_peth(
                        neural,
                        context,
                        region,
                        context["stimOn_times"],
                        base_mask,
                        group_name,
                        group_level,
                        config,
                        shared_priors,
                    )
                )
                results[region]["action"][group_level] = (
                    compute_region_grouped_population_peth(
                        neural,
                        context,
                        region,
                        context["firstMovement_times"],
                        base_mask,
                        group_name,
                        group_level,
                        config,
                        shared_priors,
                    )
                )
        except Exception as exc:
            results.pop(region, None)
            errors.append(
                make_error(error_stage, neural.eid, region, exc)
            )
            if not config.continue_on_error:
                raise

    return results, counts, pd.DataFrame(errors)


def run_hesitation_peth_analysis(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    mapped_regions: tuple[str, ...],
    config: ProcessingConfig,
) -> tuple[
    dict[str, dict[str, dict[int, PopulationPETH]]],
    pd.DataFrame,
    pd.DataFrame,
]:
    return run_grouped_peth_analysis(
        neural,
        context,
        mapped_regions,
        "hesitation_quartile",
        tuple(range(1, config.hesitation_n_quartiles + 1)),
        "hesitation_peth",
        config,
    )


def run_no_hesitation_vs_q4_peth_analysis(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    mapped_regions: tuple[str, ...],
    config: ProcessingConfig,
) -> tuple[
    dict[str, dict[str, dict[int, PopulationPETH]]],
    pd.DataFrame,
    pd.DataFrame,
]:
    return run_grouped_peth_analysis(
        neural,
        context,
        mapped_regions,
        "no_hesitation_vs_q4",
        (0, config.hesitation_n_quartiles),
        "no_hesitation_vs_q4_peth",
        config,
    )


def run_hesitation_presence_peth_analysis(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    mapped_regions: tuple[str, ...],
    config: ProcessingConfig,
) -> tuple[
    dict[str, dict[str, dict[int, PopulationPETH]]],
    pd.DataFrame,
    pd.DataFrame,
]:
    return run_grouped_peth_analysis(
        neural,
        context,
        mapped_regions,
        "has_hesitation",
        (0, 1),
        "hesitation_presence_peth",
        config,
    )


## 6. Event-aligned population activity plot


In [ ]:
# Light, high-contrast, color-vision-friendly palettes. All data traces
# are solid; condition is encoded consistently by color in both PETHs.
PETH_CONTRAST_COLORS = (
    "#858585",  # 0%: neutral reference
    "#4C9FE6",  # sky blue
    "#D9A300",  # golden yellow
    "#2FB47C",  # green
    "#D96FA6",  # rose
)

PETH_HESITATION_COLORS = (
    "#4C9FE6",  # Q1: sky blue
    "#2FB47C",  # Q2: green
    "#F0A33A",  # Q3: amber
    "#D96FA6",  # Q4: rose
)

PETH_HESITATION_STATE_COLORS = (
    "#9AA8B4",  # No hesitation: neutral blue-gray
    "#E07A5F",  # Hesitation: warm terracotta
)

PETH_NO_HESITATION_Q4_COLORS = (
    PETH_HESITATION_STATE_COLORS[0],
    PETH_HESITATION_COLORS[3],
)


def _publication_rc() -> dict[str, Any]:
    return {
        "font.family": "DejaVu Sans",
        "font.size": 9.25,
        "axes.labelsize": 10,
        "axes.titlesize": 10.25,
        "xtick.labelsize": 8.75,
        "ytick.labelsize": 8.75,
        "axes.linewidth": 0.75,
        "text.color": "#27313A",
        "axes.labelcolor": "#27313A",
        "axes.titlecolor": "#27313A",
        "axes.edgecolor": "#AEB8C2",
        "xtick.color": "#5F6B76",
        "ytick.color": "#5F6B76",
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 3.0,
        "ytick.major.size": 3.0,
        "savefig.facecolor": "white",
    }


def _safe_filename_component(value: str) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9_.-]+", "-", str(value)).strip("-._")
    return cleaned or "unknown"


def _save_figure_formats(
    fig: plt.Figure,
    output_dir: Path,
    stem: str,
    config: ProcessingConfig,
) -> list[Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    saved_paths: list[Path] = []
    for file_format in config.figure_formats:
        output_path = output_dir / f"{stem}.{file_format.lower()}"
        fig.savefig(
            output_path,
            dpi=config.figure_dpi,
            bbox_inches="tight",
            pad_inches=0.08,
            facecolor="white",
        )
        saved_paths.append(output_path.resolve())
    return saved_paths


def save_publication_figure(
    fig: plt.Figure,
    analysis: SessionAnalysis,
    region: str,
    descriptor: str,
    config: ProcessingConfig,
    analysis_path: tuple[str, ...] = (),
) -> list[Path]:
    session_name = _safe_filename_component(
        f"{analysis.subject}_{analysis.eid}"
    )
    region_name = _safe_filename_component(region)
    stem = _safe_filename_component(
        f"{analysis.subject}_{analysis.eid}_{region}_{descriptor}"
    )
    output_dir = Path(config.figure_output_dir) / session_name / region_name
    for component in analysis_path:
        output_dir /= _safe_filename_component(component)
    return _save_figure_formats(fig, output_dir, stem, config)


def save_session_figure(
    fig: plt.Figure,
    analysis: SessionAnalysis,
    descriptor: str,
    config: ProcessingConfig,
    analysis_path: tuple[str, ...] = (),
) -> list[Path]:
    session_name = _safe_filename_component(
        f"{analysis.subject}_{analysis.eid}"
    )
    stem = _safe_filename_component(
        f"{analysis.subject}_{analysis.eid}_{descriptor}"
    )
    output_dir = Path(config.figure_output_dir) / session_name
    for component in analysis_path:
        output_dir /= _safe_filename_component(component)
    return _save_figure_formats(fig, output_dir, stem, config)


def _plot_event_aligned_conditions(
    condition_results: dict[str, dict[float, PopulationPETH]],
    levels: tuple[float, ...],
    labels: tuple[str, ...],
    colors: tuple[str, ...],
    figure_title: str,
    config: ProcessingConfig,
) -> tuple[plt.Figure, np.ndarray]:
    if not (len(levels) == len(labels) == len(colors)):
        raise ValueError("PETH levels, labels, and colors must align.")
    window_ms = config.peth_pre_s * 1000
    ticks = np.linspace(-window_ms, window_ms, 5)
    tick_labels = [
        "0" if np.isclose(value, 0) else f"{value:+g}"
        for value in ticks
    ]
    panel_gap = float(
        np.clip(
            config.display_gap_s
            / (config.peth_pre_s + config.peth_post_s)
            * 0.48,
            0.09,
            0.14,
        )
    )

    with plt.rc_context(_publication_rc()):
        fig, axes = plt.subplots(
            1,
            2,
            figsize=(9.4, 3.7),
            sharey=True,
            gridspec_kw={"wspace": panel_gap},
        )
        stimulus_ax, action_ax = axes
        handles = []

        for level, label, color in zip(levels, labels, colors):
            stimulus = condition_results["stimulus"][level]
            action = condition_results["action"][level]

            line, = stimulus_ax.plot(
                stimulus.time_s * 1000,
                stimulus.mean_hz,
                color=color,
                linestyle="-",
                linewidth=2.05,
                label=label,
                zorder=3,
            )
            handles.append(line)
            stimulus_ax.fill_between(
                stimulus.time_s * 1000,
                stimulus.mean_hz - stimulus.sem_hz,
                stimulus.mean_hz + stimulus.sem_hz,
                color=color,
                alpha=0.10,
                linewidth=0,
                zorder=2,
            )
            action_ax.plot(
                action.time_s * 1000,
                action.mean_hz,
                color=color,
                linestyle="-",
                linewidth=2.05,
                zorder=3,
            )
            action_ax.fill_between(
                action.time_s * 1000,
                action.mean_hz - action.sem_hz,
                action.mean_hz + action.sem_hz,
                color=color,
                alpha=0.10,
                linewidth=0,
                zorder=2,
            )

        for ax, title in zip(
            axes,
            ("Stimulus onset", "First movement"),
        ):
            ax.axvline(
                0,
                color="#8E99A4",
                linestyle=(0, (4, 3)),
                linewidth=0.9,
                zorder=4,
            )
            ax.set_xlim(-window_ms, window_ms)
            ax.set_xticks(ticks)
            ax.set_xticklabels(tick_labels)
            ax.set_title(title, pad=7, fontweight="semibold")
            ax.yaxis.grid(
                True,
                color="#E5E9ED",
                linewidth=0.7,
                alpha=0.9,
            )
            ax.xaxis.grid(False)
            ax.set_axisbelow(True)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        stimulus_ax.set_ylabel("Population firing rate (Hz)", labelpad=8)
        stimulus_ax.spines["right"].set_visible(False)
        action_ax.spines["left"].set_visible(False)
        action_ax.tick_params(axis="y", left=False, labelleft=False)
        fig.suptitle(
            figure_title,
            x=0.10,
            y=0.975,
            ha="left",
            fontsize=12,
            fontweight="semibold",
        )
        fig.legend(
            handles=handles,
            labels=[handle.get_label() for handle in handles],
            frameon=False,
            ncol=len(handles),
            loc="upper right",
            bbox_to_anchor=(0.985, 0.955),
            columnspacing=1.0,
            handlelength=2.2,
            fontsize=8.5,
        )
        fig.supxlabel("Time from event (ms)", x=0.54, y=0.055)
        fig.subplots_adjust(
            left=0.10,
            right=0.985,
            bottom=0.19,
            top=0.76,
        )
        return fig, axes


def plot_event_aligned_population_activity(
    analysis: SessionAnalysis,
    region: str,
    config: ProcessingConfig,
) -> tuple[plt.Figure, np.ndarray]:
    if region not in analysis.peth_results:
        raise ValueError(f"No PETH result for region {region}.")
    if len(config.contrast_levels) > len(PETH_CONTRAST_COLORS):
        raise ValueError(
            "Add colors to PETH_CONTRAST_COLORS for the requested contrasts."
        )
    levels = tuple(float(value) for value in config.contrast_levels)
    return _plot_event_aligned_conditions(
        analysis.peth_results[region],
        levels,
        tuple(f"{value * 100:g}%" for value in levels),
        PETH_CONTRAST_COLORS[: len(levels)],
        f"{region} population activity",
        config,
    )


def plot_event_aligned_activity_by_hesitation(
    analysis: SessionAnalysis,
    region: str,
    config: ProcessingConfig,
) -> tuple[plt.Figure, np.ndarray]:
    if region not in analysis.hesitation_peth_results:
        raise ValueError(f"No hesitation PETH result for region {region}.")
    levels = tuple(range(1, config.hesitation_n_quartiles + 1))
    if len(levels) > len(PETH_HESITATION_COLORS):
        raise ValueError("Add colors for the requested hesitation quartiles.")
    return _plot_event_aligned_conditions(
        analysis.hesitation_peth_results[region],
        levels,
        tuple(f"Q{value}" for value in levels),
        PETH_HESITATION_COLORS[: len(levels)],
        f"{region} activity by hesitation",
        config,
    )


def plot_event_aligned_activity_by_hesitation_presence(
    analysis: SessionAnalysis,
    region: str,
    config: ProcessingConfig,
) -> tuple[plt.Figure, np.ndarray]:
    if region not in analysis.hesitation_presence_peth_results:
        raise ValueError(
            f"No hesitation-presence PETH result for region {region}."
        )
    return _plot_event_aligned_conditions(
        analysis.hesitation_presence_peth_results[region],
        (0, 1),
        ("No hesitation", "Hesitation"),
        PETH_HESITATION_STATE_COLORS,
        f"{region} activity by hesitation presence",
        config,
    )


def plot_event_aligned_activity_no_hesitation_vs_q4(
    analysis: SessionAnalysis,
    region: str,
    config: ProcessingConfig,
) -> tuple[plt.Figure, np.ndarray]:
    if region not in analysis.no_hesitation_vs_q4_peth_results:
        raise ValueError(
            f"No no-hesitation/Q4 PETH result for region {region}."
        )
    q4 = config.hesitation_n_quartiles
    return _plot_event_aligned_conditions(
        analysis.no_hesitation_vs_q4_peth_results[region],
        (0, q4),
        ("No hesitation", "Q4"),
        PETH_NO_HESITATION_Q4_COLORS,
        f"{region} activity: no hesitation vs Q4",
        config,
    )


def plot_hesitation_score_quartiles(
    analysis: SessionAnalysis,
    config: ProcessingConfig,
) -> tuple[plt.Figure, np.ndarray]:
    required = {
        "trial_id",
        "hesitation_score",
        "has_hesitation",
        "hesitation_quartile",
    }
    full_table = analysis.hesitation.trial_table.copy()
    missing = required - set(full_table.columns)
    if missing:
        raise ValueError(
            f"Hesitation trial table is missing: {sorted(missing)}"
        )
    score_table = full_table.dropna(
        subset=["hesitation_score", "has_hesitation"]
    ).copy()
    if score_table.empty:
        raise ValueError("No valid hesitation scores are available.")
    score_table["has_hesitation"] = score_table[
        "has_hesitation"
    ].astype(int)

    table = full_table.dropna(
        subset=["hesitation_score", "hesitation_quartile"]
    ).copy()
    if table.empty:
        raise ValueError("No positive hesitation scores are available.")
    table["hesitation_quartile"] = table[
        "hesitation_quartile"
    ].astype(int)
    expected = set(range(1, config.hesitation_n_quartiles + 1))
    observed = set(table["hesitation_quartile"].unique())
    if observed != expected:
        raise ValueError(
            f"Expected hesitation quartiles {sorted(expected)}, "
            f"found {sorted(observed)}."
        )

    scores = score_table["hesitation_score"].to_numpy(dtype=float)
    score_values, score_counts = np.unique(scores, return_counts=True)
    ordered = table.sort_values(
        ["hesitation_quartile", "hesitation_score", "trial_id"],
        kind="stable",
    ).reset_index(drop=True)
    rank_x = np.arange(1, len(ordered) + 1)

    with plt.rc_context(_publication_rc()):
        fig, axes = plt.subplots(
            1,
            2,
            figsize=(10.4, 3.8),
            gridspec_kw={"width_ratios": (0.86, 1.55), "wspace": 0.25},
        )
        distribution_ax, quartile_ax = axes

        distribution_ax.bar(
            score_values,
            score_counts,
            width=0.78,
            color=[
                (
                    PETH_HESITATION_STATE_COLORS[0]
                    if np.isclose(value, 0.0)
                    else PETH_HESITATION_STATE_COLORS[1]
                )
                for value in score_values
            ],
            edgecolor="white",
            linewidth=0.7,
            zorder=3,
        )
        distribution_ax.set_title(
            "All valid trials", pad=7, fontweight="semibold"
        )
        distribution_ax.set_xlabel("Direction changes per trial", labelpad=7)
        distribution_ax.set_ylabel("Trials", labelpad=7)
        distribution_ax.locator_params(axis="x", integer=True, nbins=7)
        distribution_ax.locator_params(axis="y", integer=True, nbins=6)
        state_counts = score_table["has_hesitation"].value_counts()
        distribution_ax.text(
            0.97,
            0.95,
            (
                f"No hesitation  {int(state_counts.get(0, 0))}\n"
                f"Hesitation  {int(state_counts.get(1, 0))}"
            ),
            transform=distribution_ax.transAxes,
            ha="right",
            va="top",
            fontsize=8.3,
            color="#52606C",
            linespacing=1.35,
            bbox={
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 0.86,
                "pad": 2.5,
            },
        )

        quartile_ax.plot(
            rank_x,
            ordered["hesitation_score"],
            color="#C5CDD5",
            linewidth=0.85,
            zorder=1,
        )
        group_start = 1
        for quartile, color in zip(
            range(1, config.hesitation_n_quartiles + 1),
            PETH_HESITATION_COLORS,
        ):
            mask = ordered["hesitation_quartile"].eq(quartile).to_numpy()
            group_x = rank_x[mask]
            group_scores = ordered.loc[mask, "hesitation_score"]
            if not len(group_x):
                continue
            group_end = int(group_x[-1])
            quartile_ax.axvspan(
                group_start - 0.5,
                group_end + 0.5,
                color=color,
                alpha=0.055,
                linewidth=0,
                zorder=0,
            )
            quartile_ax.scatter(
                group_x,
                group_scores,
                s=13 if len(ordered) <= 600 else 8,
                color=color,
                edgecolor="none",
                alpha=0.82,
                zorder=3,
            )
            quartile_ax.text(
                (group_start + group_end) / 2,
                0.97,
                f"Q{quartile}",
                transform=quartile_ax.get_xaxis_transform(),
                color=color,
                ha="center",
                va="top",
                fontsize=9,
                fontweight="semibold",
            )
            if quartile < config.hesitation_n_quartiles:
                quartile_ax.axvline(
                    group_end + 0.5,
                    color="#9DA8B2",
                    linestyle=(0, (3, 2)),
                    linewidth=0.85,
                    zorder=2,
                )
            group_start = group_end + 1

        quartile_ax.set_title(
            "Positive-score quartiles", pad=7, fontweight="semibold"
        )
        quartile_ax.set_xlabel("Trials ordered by score", labelpad=7)
        quartile_ax.set_ylabel("Hesitation score", labelpad=7)
        quartile_ax.set_xlim(0.5, len(ordered) + 0.5)
        quartile_ax.locator_params(axis="y", integer=True, nbins=7)

        for axis in axes:
            axis.yaxis.grid(
                True,
                color="#E5E9ED",
                linewidth=0.7,
                alpha=0.9,
            )
            axis.xaxis.grid(False)
            axis.set_axisbelow(True)
            axis.spines["top"].set_visible(False)
            axis.spines["right"].set_visible(False)

        fig.suptitle(
            "Hesitation scores",
            x=0.075,
            y=0.985,
            ha="left",
            fontsize=12,
            fontweight="semibold",
        )
        fig.subplots_adjust(
            left=0.075,
            right=0.985,
            bottom=0.19,
            top=0.78,
        )
        return fig, axes


## 7. Efficient firing-rate features and Linear-SVM decoding

The original nested loop repeatedly scanned all spikes for every trial. Here, sorted
spikes are sliced with `searchsorted` and counted with `bincount`, preserving the same
firing-rate definition while scaling much better.


In [ ]:
def compute_window_firing_rate(
    spike_times: np.ndarray,
    spike_units: np.ndarray,
    unit_ids: np.ndarray,
    event_times: np.ndarray,
    trial_ids: np.ndarray,
    start_offset: float,
    end_offset: float,
) -> np.ndarray:
    if end_offset <= start_offset:
        raise ValueError("end_offset must be greater than start_offset.")
    unit_ids = np.asarray(unit_ids, dtype=int)
    if not len(unit_ids):
        raise ValueError("unit_ids cannot be empty.")

    lookup_size = int(max(spike_units.max(), unit_ids.max())) + 1
    lookup = np.full(lookup_size, -1, dtype=int)
    lookup[unit_ids] = np.arange(len(unit_ids))
    output = np.zeros((len(trial_ids), len(unit_ids)), dtype=float)
    duration = end_offset - start_offset

    for row, trial_id in enumerate(trial_ids):
        start = event_times[trial_id] + start_offset
        end = event_times[trial_id] + end_offset
        left = np.searchsorted(spike_times, start, side="left")
        right = np.searchsorted(spike_times, end, side="left")
        local = lookup[spike_units[left:right]]
        local = local[local >= 0]
        output[row] = np.bincount(local, minlength=len(unit_ids)) / duration
    return output


def decoder_trial_data(
    context: dict[str, np.ndarray],
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    valid = (
        np.isfinite(context["stimOn_times"])
        & np.isfinite(context["firstMovement_times"])
        & np.isfinite(context["choice"])
        & np.isin(context["choice"], [-1.0, 1.0])
    )
    trial_ids = np.flatnonzero(valid)
    y = (context["choice"][trial_ids] == 1).astype(int)
    contrasts = context["decoder_contrast"][trial_ids]
    return trial_ids, y, contrasts


def build_decoder_features(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    region: str,
    config: ProcessingConfig,
) -> tuple[dict[str, np.ndarray], np.ndarray, np.ndarray]:
    trial_ids, y, contrasts = decoder_trial_data(context)
    unit_ids = neural.units.loc[
        neural.units["region"].eq(region), "unit_id"
    ].to_numpy(dtype=int)
    window = config.decoder_window_s

    features = {
        "Pre-stimulus": compute_window_firing_rate(
            neural.spike_times,
            neural.spike_units,
            unit_ids,
            context["stimOn_times"],
            trial_ids,
            -window,
            0.0,
        ),
        "Post-stimulus": compute_window_firing_rate(
            neural.spike_times,
            neural.spike_units,
            unit_ids,
            context["stimOn_times"],
            trial_ids,
            0.0,
            window,
        ),
        "Pre-action": compute_window_firing_rate(
            neural.spike_times,
            neural.spike_units,
            unit_ids,
            context["firstMovement_times"],
            trial_ids,
            -window,
            0.0,
        ),
        "Post-action": compute_window_firing_rate(
            neural.spike_times,
            neural.spike_units,
            unit_ids,
            context["firstMovement_times"],
            trial_ids,
            0.0,
            window,
        ),
    }
    return features, y, contrasts


def cross_validated_svm_predictions(
    X: np.ndarray,
    y: np.ndarray,
    config: ProcessingConfig,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    classes, counts = np.unique(y, return_counts=True)
    if len(classes) != 2:
        raise ValueError(f"Decoder requires two choice classes; found {classes.tolist()}.")
    if counts.min() < config.decoder_n_splits:
        raise ValueError(
            f"Smallest choice class has {counts.min()} trials, fewer than "
            f"decoder_n_splits={config.decoder_n_splits}."
        )

    splitter = StratifiedKFold(
        n_splits=config.decoder_n_splits,
        shuffle=True,
        random_state=config.random_state,
    )
    fold_rows: list[dict] = []
    prediction_rows: list[dict] = []

    for fold, (train_index, test_index) in enumerate(splitter.split(X, y)):
        model = Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "svm",
                    LinearSVC(
                        class_weight="balanced",
                        random_state=config.random_state,
                        max_iter=config.decoder_max_iter,
                    ),
                ),
            ]
        )
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            model.fit(X[train_index], y[train_index])
        prediction = model.predict(X[test_index])
        y_test = y[test_index]

        fold_rows.append(
            {
                "fold": fold,
                "accuracy": accuracy_score(y_test, prediction),
                "balanced_accuracy": balanced_accuracy_score(y_test, prediction),
            }
        )
        prediction_rows.extend(
            {
                "fold": fold,
                "sample_index": int(sample_index),
                "true_choice": int(truth),
                "predicted_choice": int(predicted),
            }
            for sample_index, truth, predicted in zip(
                test_index, y_test, prediction
            )
        )

    return pd.DataFrame(fold_rows), pd.DataFrame(prediction_rows)


def summarize_decoder_accuracy_by_group(
    predictions: pd.DataFrame,
    group_values: np.ndarray,
    group_name: str,
) -> pd.DataFrame:
    group_values = np.asarray(group_values, dtype=float)
    if predictions.empty:
        return pd.DataFrame(
            columns=["fold", group_name, "accuracy", "n_trials"]
        )
    sample_ids = predictions["sample_index"].to_numpy(dtype=int)
    if sample_ids.min() < 0 or sample_ids.max() >= len(group_values):
        raise ValueError("Decoder prediction sample index is out of bounds.")

    grouped = predictions.copy()
    grouped[group_name] = group_values[sample_ids]
    grouped = grouped[np.isfinite(grouped[group_name])]
    rows: list[dict] = []
    for (fold, level), frame in grouped.groupby(
        ["fold", group_name], sort=True
    ):
        rows.append(
            {
                "fold": int(fold),
                group_name: float(level),
                "accuracy": accuracy_score(
                    frame["true_choice"], frame["predicted_choice"]
                ),
                "n_trials": len(frame),
            }
        )
    return pd.DataFrame(
        rows,
        columns=["fold", group_name, "accuracy", "n_trials"],
    )


def run_svm_cross_validation(
    X: np.ndarray,
    y: np.ndarray,
    contrasts: np.ndarray,
    config: ProcessingConfig,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Compatibility wrapper for the original contrast-grouped output.'''
    folds, predictions = cross_validated_svm_predictions(X, y, config)
    contrast_rows = summarize_decoder_accuracy_by_group(
        predictions,
        contrasts,
        "contrast",
    )
    return folds, contrast_rows


def run_decoder_analysis(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    mapped_regions: tuple[str, ...],
    config: ProcessingConfig,
) -> tuple[
    dict[str, dict[str, pd.DataFrame]],
    pd.DataFrame,
    pd.DataFrame,
    dict[str, tuple[dict[str, np.ndarray], np.ndarray, np.ndarray]],
]:
    results: dict[str, dict[str, pd.DataFrame]] = {}
    summary_rows: list[dict] = []
    errors: list[dict] = []
    feature_cache: dict[
        str,
        tuple[dict[str, np.ndarray], np.ndarray, np.ndarray],
    ] = {}

    for region in mapped_regions:
        try:
            features, y, contrasts = build_decoder_features(
                neural, context, region, config
            )
            trial_ids, _, _ = decoder_trial_data(context)
            hesitation_quartiles = np.asarray(
                context.get(
                    "hesitation_quartile",
                    np.full(len(context["choice"]), np.nan),
                ),
                dtype=float,
            )[trial_ids]
            hesitation_presence = np.asarray(
                context.get(
                    "has_hesitation",
                    np.full(len(context["choice"]), np.nan),
                ),
                dtype=float,
            )[trial_ids]
            no_hesitation_vs_q4 = np.asarray(
                context.get(
                    "no_hesitation_vs_q4",
                    np.full(len(context["choice"]), np.nan),
                ),
                dtype=float,
            )[trial_ids]
            feature_cache[region] = (features, y, contrasts)
            phase_fold_parts: list[pd.DataFrame] = []
            phase_contrast_parts: list[pd.DataFrame] = []
            phase_hesitation_parts: list[pd.DataFrame] = []
            phase_hesitation_presence_parts: list[pd.DataFrame] = []
            phase_no_hesitation_vs_q4_parts: list[pd.DataFrame] = []

            for phase, X in features.items():
                fold_df, predictions = cross_validated_svm_predictions(
                    X, y, config
                )
                contrast_df = summarize_decoder_accuracy_by_group(
                    predictions, contrasts, "contrast"
                )
                hesitation_df = summarize_decoder_accuracy_by_group(
                    predictions,
                    hesitation_quartiles,
                    "hesitation_quartile",
                )
                hesitation_presence_df = (
                    summarize_decoder_accuracy_by_group(
                        predictions,
                        hesitation_presence,
                        "has_hesitation",
                    )
                )
                no_hesitation_vs_q4_df = (
                    summarize_decoder_accuracy_by_group(
                        predictions,
                        no_hesitation_vs_q4,
                        "no_hesitation_vs_q4",
                    )
                )
                fold_df.insert(0, "phase", phase)
                contrast_df.insert(0, "phase", phase)
                hesitation_df.insert(0, "phase", phase)
                hesitation_presence_df.insert(0, "phase", phase)
                no_hesitation_vs_q4_df.insert(0, "phase", phase)
                phase_fold_parts.append(fold_df)
                phase_contrast_parts.append(contrast_df)
                phase_hesitation_parts.append(hesitation_df)
                phase_hesitation_presence_parts.append(
                    hesitation_presence_df
                )
                phase_no_hesitation_vs_q4_parts.append(
                    no_hesitation_vs_q4_df
                )
                summary_rows.append(
                    {
                        "eid": neural.eid,
                        "subject": neural.subject,
                        "region": region,
                        "phase": phase,
                        "n_trials": len(y),
                        "n_units": X.shape[1],
                        "accuracy": fold_df["accuracy"].mean(),
                        "accuracy_sd": fold_df["accuracy"].std(),
                        "balanced_accuracy": fold_df[
                            "balanced_accuracy"
                        ].mean(),
                    }
                )

            results[region] = {
                "folds": pd.concat(phase_fold_parts, ignore_index=True),
                "contrast": pd.concat(
                    phase_contrast_parts, ignore_index=True
                ),
                "hesitation": pd.concat(
                    phase_hesitation_parts, ignore_index=True
                ),
                "hesitation_presence": pd.concat(
                    phase_hesitation_presence_parts,
                    ignore_index=True,
                ),
                "no_hesitation_vs_q4": pd.concat(
                    phase_no_hesitation_vs_q4_parts,
                    ignore_index=True,
                ),
            }
        except Exception as exc:
            errors.append(make_error("decoder", neural.eid, region, exc))
            if not config.continue_on_error:
                raise

    return (
        results,
        pd.DataFrame(summary_rows),
        pd.DataFrame(errors),
        feature_cache,
    )


## 8. Comparative four-phase decoder plot


In [ ]:
DECODER_PHASES = [
    "Pre-stimulus",
    "Post-stimulus",
    "Pre-action",
    "Post-action",
]

DECODER_LINE_COLOR = "#4C9FE6"


def _plot_decoder_accuracy_by_group(
    grouped_accuracy: pd.DataFrame,
    group_name: str,
    levels: np.ndarray,
    tick_labels: tuple[str, ...],
    figure_title: str,
    x_label: str,
    region: str,
    config: ProcessingConfig,
) -> tuple[plt.Figure, np.ndarray]:
    levels = np.asarray(levels, dtype=float)
    if len(levels) != len(tick_labels):
        raise ValueError("Decoder levels and tick labels must align.")
    x = np.arange(len(levels))
    window_ms = int(round(config.decoder_window_s * 1000))
    phase_titles = {
        "Pre-stimulus": f"Pre-stimulus\n−{window_ms}–0 ms",
        "Post-stimulus": f"Post-stimulus\n0–{window_ms} ms",
        "Pre-action": f"Pre-action\n−{window_ms}–0 ms",
        "Post-action": f"Post-action\n0–{window_ms} ms",
    }

    with plt.rc_context(_publication_rc()):
        fig, axes = plt.subplots(
            1,
            4,
            figsize=(11.4, 3.7),
            sharey=True,
        )
        fig.subplots_adjust(
            left=0.075,
            right=0.985,
            bottom=0.20,
            top=0.72,
            wspace=0.0,
        )

        for axis, phase in zip(axes, DECODER_PHASES):
            phase_data = grouped_accuracy[
                grouped_accuracy["phase"].eq(phase)
            ]
            if phase_data.empty:
                raise ValueError(f"No decoder result for {region} / {phase}.")
            summary = (
                phase_data.groupby(group_name)["accuracy"]
                .agg(["mean", "std", "count"])
                .reindex(levels)
            )
            if summary["mean"].isna().any():
                missing = levels[summary["mean"].isna()].tolist()
                raise ValueError(
                    f"Missing {group_name} decoder levels for {region} / "
                    f"{phase}: {missing}"
                )
            sem = summary["std"] / np.sqrt(summary["count"])
            color = DECODER_LINE_COLOR
            axis.errorbar(
                x,
                summary["mean"],
                yerr=sem,
                color=color,
                marker="o",
                markerfacecolor=color,
                markeredgecolor="white",
                markeredgewidth=0.8,
                markersize=5.8,
                linewidth=2.05,
                elinewidth=1.05,
                capsize=2.8,
                capthick=1.05,
                zorder=3,
            )
            axis.axhline(
                0.5,
                color="#AEB7C0",
                linestyle=(0, (4, 3)),
                linewidth=0.9,
                zorder=2,
            )
            axis.set_xticks(x)
            axis.set_xticklabels(tick_labels)
            axis.set_xlim(-0.35, len(levels) - 0.35)
            axis.set_ylim(0.4, 1.0)
            axis.set_yticks(np.arange(0.4, 1.01, 0.1))
            axis.set_title(
                phase_titles[phase],
                pad=7,
                fontweight="semibold",
                linespacing=1.25,
            )
            axis.yaxis.grid(
                True,
                color="#E5E9ED",
                linewidth=0.7,
                alpha=0.9,
            )
            axis.xaxis.grid(False)
            axis.set_axisbelow(True)
            axis.spines["top"].set_visible(False)
            axis.spines["right"].set_visible(False)

        for axis in axes[1:]:
            axis.spines["left"].set_visible(False)
            axis.tick_params(axis="y", left=False, labelleft=False)
        axes[0].set_ylabel("Accuracy", labelpad=7)
        fig.suptitle(
            figure_title,
            x=0.075,
            y=0.97,
            ha="left",
            fontsize=12,
            fontweight="semibold",
        )
        fig.supxlabel(x_label, x=0.53, y=0.055)
        return fig, axes


def plot_decoder_accuracy_by_contrast(
    analysis: SessionAnalysis,
    region: str,
    config: ProcessingConfig,
) -> tuple[plt.Figure, np.ndarray]:
    if region not in analysis.decoder_results:
        raise ValueError(f"No decoder result for region {region}.")
    levels = np.asarray(config.contrast_levels, dtype=float)
    return _plot_decoder_accuracy_by_group(
        analysis.decoder_results[region]["contrast"],
        "contrast",
        levels,
        tuple(f"{value * 100:g}" for value in levels),
        f"{region} choice decoding",
        "Visual contrast (%)",
        region,
        config,
    )


def plot_decoder_accuracy_by_hesitation(
    analysis: SessionAnalysis,
    region: str,
    config: ProcessingConfig,
) -> tuple[plt.Figure, np.ndarray]:
    if region not in analysis.decoder_results:
        raise ValueError(f"No decoder result for region {region}.")
    levels = np.arange(1, config.hesitation_n_quartiles + 1, dtype=float)
    return _plot_decoder_accuracy_by_group(
        analysis.decoder_results[region]["hesitation"],
        "hesitation_quartile",
        levels,
        tuple(f"Q{int(value)}" for value in levels),
        f"{region} choice decoding by hesitation",
        "Hesitation-score quartile",
        region,
        config,
    )


def plot_decoder_accuracy_no_hesitation_vs_q4(
    analysis: SessionAnalysis,
    region: str,
    config: ProcessingConfig,
) -> tuple[plt.Figure, np.ndarray]:
    if region not in analysis.decoder_results:
        raise ValueError(f"No decoder result for region {region}.")
    return _plot_decoder_accuracy_by_group(
        analysis.decoder_results[region]["no_hesitation_vs_q4"],
        "no_hesitation_vs_q4",
        np.asarray([0.0, float(config.hesitation_n_quartiles)]),
        ("No hesitation", "Q4"),
        f"{region} choice decoding: no hesitation vs Q4",
        "Hesitation group",
        region,
        config,
    )


def plot_decoder_accuracy_by_hesitation_presence(
    analysis: SessionAnalysis,
    region: str,
    config: ProcessingConfig,
) -> tuple[plt.Figure, np.ndarray]:
    if region not in analysis.decoder_results:
        raise ValueError(f"No decoder result for region {region}.")
    return _plot_decoder_accuracy_by_group(
        analysis.decoder_results[region]["hesitation_presence"],
        "has_hesitation",
        np.asarray([0.0, 1.0]),
        ("Absent", "Present"),
        f"{region} choice decoding by hesitation presence",
        "Hesitation",
        region,
        config,
    )


## 9. Event-window PCA (four neural states)

A single scaler and PCA basis are fitted per `session × region` on the four stacked
firing-rate matrices. Each phase is then transformed separately in that shared basis,
so the four points of every trial can be connected meaningfully.

Each contrast receives separate 2D and 3D publication figures. Right and left choices
remain overlaid on the same axis, and every contrast-specific figure uses the same global
PC limits (plus the same camera angle in 3D) for valid comparison. The displayed path is
a shape-preserving PCHIP curve that passes through all four measured state centroids;
smoothing is visual only. The full phase order is connected as
`pre-stimulus → post-stimulus → pre-action → post-action`, with an arrow on every
transition. Choice is redundantly encoded by line style and marker fill. Individual
trial coordinates remain in `scores`, and a bounded sample can be enabled in `CONFIG`.


In [ ]:
PCA_PHASES = tuple(DECODER_PHASES)
PCA_PC_COLUMNS = ("PC1", "PC2", "PC3")
PCA_PHASE_MARKERS = ("o", "^", "s", "D")
PCA_PHASE_LABELS = (
    "Pre-stimulus",
    "Post-stimulus",
    "Pre-action",
    "Post-action",
)
PCA_CONTRAST_COLORS = (
    "#222222",  # 0%: black
    "#4C9FE6",  # 6.25%: blue
    "#D9A300",  # 12.5%: visible golden yellow
    "#2FB47C",  # 25%: green
    "#D96FA6",  # 100%: rose, required for the fifth contrast
)
PCA_CHOICE_LINESTYLES = {
    "Right": "-",
    "Left": (0, (6.0, 3.0)),
}


def _pca_contrast_color(
    contrast: float,
    config: ProcessingConfig,
) -> str:
    if len(config.contrast_levels) > len(PCA_CONTRAST_COLORS):
        raise ValueError(
            "Add colors to PCA_CONTRAST_COLORS for the requested contrasts."
        )
    for level, color in zip(
        config.contrast_levels,
        PCA_CONTRAST_COLORS,
    ):
        if np.isclose(contrast, level):
            return color
    raise ValueError(
        f"Contrast {contrast:g} is not present in config.contrast_levels."
    )


def _trajectory_display_curve(
    coordinates: np.ndarray,
    *,
    smooth: bool,
    samples_per_segment: int = 12,
) -> tuple[np.ndarray, np.ndarray]:
    '''Densify a trajectory for rendering without changing its anchors.'''
    coordinates = np.asarray(coordinates, dtype=float)
    if coordinates.ndim != 2 or coordinates.shape[0] < 2:
        raise ValueError(
            "Trajectory coordinates must contain at least two points."
        )
    if samples_per_segment < 2:
        raise ValueError("samples_per_segment must be at least 2.")
    if not np.all(np.isfinite(coordinates)):
        raise ValueError("Trajectory coordinates must be finite.")

    source_parameter = np.arange(len(coordinates), dtype=float)
    display_parameter = np.linspace(
        source_parameter[0],
        source_parameter[-1],
        (len(coordinates) - 1) * samples_per_segment + 1,
    )
    if smooth and len(coordinates) >= 3:
        display_coordinates = PchipInterpolator(
            source_parameter,
            coordinates,
            axis=0,
        )(display_parameter)
    else:
        display_coordinates = np.column_stack(
            [
                np.interp(
                    display_parameter,
                    source_parameter,
                    coordinates[:, dimension],
                )
                for dimension in range(coordinates.shape[1])
            ]
        )
    return display_parameter, np.asarray(display_coordinates, dtype=float)


def _trajectory_arrow_segment(
    display_parameter: np.ndarray,
    display_coordinates: np.ndarray,
    stage_start: float,
    stage_end: float,
) -> tuple[np.ndarray, np.ndarray] | None:
    '''Return a short local tangent segment near the middle of a stage.'''
    display_parameter = np.asarray(display_parameter, dtype=float)
    display_coordinates = np.asarray(display_coordinates, dtype=float)
    if stage_end <= stage_start:
        raise ValueError("Trajectory arrow stage must have positive length.")
    if len(display_parameter) != len(display_coordinates):
        raise ValueError("Trajectory parameters and coordinates do not align.")

    span = stage_end - stage_start
    for start_fraction, end_fraction in (
        (0.54, 0.72),
        (0.46, 0.78),
        (0.32, 0.84),
    ):
        query = np.asarray(
            [
                stage_start + start_fraction * span,
                stage_start + end_fraction * span,
            ]
        )
        segment = np.column_stack(
            [
                np.interp(
                    query,
                    display_parameter,
                    display_coordinates[:, dimension],
                )
                for dimension in range(display_coordinates.shape[1])
            ]
        )
        if np.linalg.norm(segment[1] - segment[0]) > np.finfo(float).eps:
            return segment[0], segment[1]
    return None


def _style_trajectory_line(line: Line2D, choice: str) -> None:
    '''Use rounded joins and dash caps for clean projected trajectories.'''
    if choice not in PCA_CHOICE_LINESTYLES:
        raise ValueError(f"Unknown PCA choice {choice!r}.")
    line.set_antialiased(True)
    line.set_solid_capstyle("round")
    line.set_solid_joinstyle("round")
    line.set_dash_capstyle("round")
    line.set_dash_joinstyle("round")


class _TrajectoryArrow3D(FancyArrowPatch):
    '''Screen-stable 3D arrow projected from two PCA coordinates.'''

    def __init__(
        self,
        start: np.ndarray,
        end: np.ndarray,
        **kwargs: Any,
    ) -> None:
        super().__init__((0.0, 0.0), (0.0, 0.0), **kwargs)
        start = np.asarray(start, dtype=float)
        end = np.asarray(end, dtype=float)
        if start.shape != (3,) or end.shape != (3,):
            raise ValueError("A 3D trajectory arrow requires two 3D points.")
        self._vertices_3d = np.vstack([start, end])

    def _project(self) -> np.ndarray:
        projected = np.asarray(
            proj3d.proj_transform(
                self._vertices_3d[:, 0],
                self._vertices_3d[:, 1],
                self._vertices_3d[:, 2],
                self.axes.get_proj(),
            )
        )
        self.set_positions(
            (projected[0, 0], projected[1, 0]),
            (projected[0, 1], projected[1, 1]),
        )
        return projected[2]

    def draw(self, renderer: Any) -> None:
        self._project()
        super().draw(renderer)

    def do_3d_projection(self, renderer: Any | None = None) -> float:
        return float(np.min(self._project()))


def fit_trial_pca(
    phase_features: dict[str, np.ndarray],
    trial_ids: np.ndarray,
    choice_values: np.ndarray,
    contrasts: np.ndarray,
    config: ProcessingConfig,
) -> PCATrajectoryResult:
    missing = [phase for phase in PCA_PHASES if phase not in phase_features]
    extra = [phase for phase in phase_features if phase not in PCA_PHASES]
    if missing or extra:
        raise ValueError(
            f"PCA phases do not match the decoder phases; missing={missing}, "
            f"extra={extra}."
        )

    trial_ids = np.asarray(trial_ids, dtype=int)
    choice_values = np.asarray(choice_values, dtype=int)
    contrasts = np.asarray(contrasts, dtype=float)
    n_trials = len(trial_ids)
    if not (
        len(choice_values) == n_trials
        and len(contrasts) == n_trials
    ):
        raise ValueError("Trial IDs, choices, and contrasts must have equal length.")
    if n_trials < 1:
        raise ValueError("PCA requires at least one valid trial.")
    if not np.all(np.isin(choice_values, [-1, 1])):
        raise ValueError("PCA choices must contain only IBL values -1 and +1.")
    if not np.all(np.isfinite(contrasts)):
        raise ValueError("PCA contrasts must be finite for every valid trial.")

    matrices: dict[str, np.ndarray] = {}
    reference_shape: tuple[int, int] | None = None
    for phase in PCA_PHASES:
        matrix = np.asarray(phase_features[phase], dtype=float)
        if matrix.ndim != 2:
            raise ValueError(f"{phase}: feature matrix must be two-dimensional.")
        if reference_shape is None:
            reference_shape = matrix.shape
        elif matrix.shape != reference_shape:
            raise ValueError(
                f"{phase}: expected feature shape {reference_shape}, "
                f"received {matrix.shape}."
            )
        if matrix.shape[0] != n_trials:
            raise ValueError(
                f"{phase}: feature rows do not match the valid trials."
            )
        if not np.all(np.isfinite(matrix)):
            raise ValueError(f"{phase}: feature matrix contains non-finite values.")
        matrices[phase] = matrix

    assert reference_shape is not None
    n_units = reference_shape[1]
    if min(4 * n_trials, n_units) < config.pca_n_components:
        raise ValueError(
            "Not enough trials or units for the requested PCA components."
        )

    combined = np.vstack([matrices[phase] for phase in PCA_PHASES])
    scaler = StandardScaler()
    combined_scaled = scaler.fit_transform(combined)
    if np.linalg.matrix_rank(combined_scaled) < config.pca_n_components:
        raise ValueError(
            "The standardized population activity has rank below three."
        )

    model = PCA(
        n_components=config.pca_n_components,
        svd_solver="full",
    )
    model.fit(combined_scaled)

    # IBL convention: +1 is a clockwise/right choice; -1 is left.
    choice_labels = np.where(choice_values == 1, "Right", "Left")
    score_frames: list[pd.DataFrame] = []
    for phase_index, phase in enumerate(PCA_PHASES):
        # Separate transform, using the one shared scaler and PCA basis.
        coordinates = model.transform(
            scaler.transform(matrices[phase])
        )
        frame = pd.DataFrame(coordinates, columns=PCA_PC_COLUMNS)
        frame.insert(0, "contrast", contrasts)
        frame.insert(0, "choice", choice_labels)
        frame.insert(0, "choice_value", choice_values)
        frame.insert(0, "phase_index", phase_index)
        frame.insert(0, "phase", phase)
        frame.insert(0, "trial_id", trial_ids)
        score_frames.append(frame)

    scores = pd.concat(score_frames, ignore_index=True)
    centroids = (
        scores.groupby(
            [
                "contrast",
                "choice",
                "choice_value",
                "phase",
                "phase_index",
            ],
            as_index=False,
            sort=False,
        )
        .agg(
            n_trials=("trial_id", "nunique"),
            PC1=("PC1", "mean"),
            PC2=("PC2", "mean"),
            PC3=("PC3", "mean"),
        )
    )
    return PCATrajectoryResult(
        scaler=scaler,
        model=model,
        scores=scores,
        centroids=centroids,
        explained_variance_ratio=model.explained_variance_ratio_.copy(),
        n_trials=n_trials,
        n_units=n_units,
    )


def run_pca_analysis(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    mapped_regions: tuple[str, ...],
    config: ProcessingConfig,
    feature_cache: dict[
        str,
        tuple[dict[str, np.ndarray], np.ndarray, np.ndarray],
    ] | None = None,
) -> tuple[dict[str, PCATrajectoryResult], pd.DataFrame, pd.DataFrame]:
    results: dict[str, PCATrajectoryResult] = {}
    summary_rows: list[dict] = []
    errors: list[dict] = []

    trial_ids, decoder_choices, trial_contrasts = decoder_trial_data(context)
    choice_values = context["choice"][trial_ids].astype(int)
    expected_decoder_choices = (choice_values == 1).astype(int)
    if not np.array_equal(decoder_choices, expected_decoder_choices):
        raise RuntimeError("Choice encoding is inconsistent with decoder_trial_data.")

    for region in mapped_regions:
        try:
            if feature_cache is not None and region in feature_cache:
                phase_features, y, contrasts = feature_cache[region]
            else:
                phase_features, y, contrasts = build_decoder_features(
                    neural,
                    context,
                    region,
                    config,
                )
            if not np.array_equal(y, decoder_choices):
                raise RuntimeError(f"{region}: PCA and SVM trial choices differ.")
            if not np.allclose(
                contrasts,
                trial_contrasts,
                equal_nan=True,
            ):
                raise RuntimeError(f"{region}: PCA and SVM trial contrasts differ.")

            result = fit_trial_pca(
                phase_features,
                trial_ids,
                choice_values,
                contrasts,
                config,
            )
            results[region] = result
            variance = result.explained_variance_ratio
            summary_rows.append(
                {
                    "eid": neural.eid,
                    "subject": neural.subject,
                    "region": region,
                    "n_trials": result.n_trials,
                    "n_units": result.n_units,
                    "pc1_variance": variance[0],
                    "pc2_variance": variance[1],
                    "pc3_variance": variance[2],
                    "cumulative_variance": variance.sum(),
                }
            )
        except Exception as exc:
            errors.append(make_error("pca", neural.eid, region, exc))
            if not config.continue_on_error:
                raise

    return results, pd.DataFrame(summary_rows), pd.DataFrame(errors)


def _draw_pca_direction_arrow(
    axis: Any,
    start: np.ndarray,
    end: np.ndarray,
    color: str,
    gid: str,
    mutation_scale: float = 10.5,
) -> Any | None:
    start = np.asarray(start, dtype=float)
    end = np.asarray(end, dtype=float)
    delta = end - start
    if not np.all(np.isfinite(delta)):
        raise ValueError("PCA direction arrow contains non-finite values.")
    if np.linalg.norm(delta) <= np.finfo(float).eps:
        return None

    arrow = _TrajectoryArrow3D(
        start,
        end,
        arrowstyle="-|>",
        mutation_scale=mutation_scale,
        linewidth=1.65,
        facecolor=color,
        edgecolor=color,
        shrinkA=0.0,
        shrinkB=0.0,
        capstyle="round",
        joinstyle="round",
        zorder=7,
    )
    axis.add_artist(arrow)
    arrow.set_gid(gid)
    arrow.set_path_effects(
        [
            path_effects.Stroke(
                linewidth=4.0,
                foreground="white",
                alpha=0.96,
            ),
            path_effects.Normal(),
        ]
    )
    return arrow


def _draw_pca_direction_arrow_2d(
    axis: Any,
    start: np.ndarray,
    end: np.ndarray,
    color: str,
    gid: str,
    mutation_scale: float = 12.5,
) -> Any | None:
    start = np.asarray(start, dtype=float)
    end = np.asarray(end, dtype=float)
    delta = end - start
    if not np.all(np.isfinite(delta)):
        raise ValueError("PCA direction arrow contains non-finite values.")
    if np.linalg.norm(delta) <= np.finfo(float).eps:
        return None

    arrow = FancyArrowPatch(
        tuple(start),
        tuple(end),
        arrowstyle="-|>",
        mutation_scale=mutation_scale,
        linewidth=1.55,
        facecolor=color,
        edgecolor=color,
        shrinkA=0.0,
        shrinkB=0.0,
        capstyle="round",
        joinstyle="round",
        zorder=7,
    )
    axis.add_patch(arrow)
    arrow.set_gid(gid)
    arrow.set_path_effects(
        [
            path_effects.Stroke(
                linewidth=3.5,
                foreground="white",
                alpha=0.96,
            ),
            path_effects.Normal(),
        ]
    )
    return arrow


def _event_window_pca_plot_inputs(
    analysis: SessionAnalysis,
    region: str,
    contrast: float,
    config: ProcessingConfig,
) -> tuple[
    PCATrajectoryResult,
    float,
    int,
    str,
    dict[str, tuple[float, float]],
]:
    if region not in analysis.pca_results:
        raise ValueError(f"No event-window PCA result for region {region}.")
    contrast = float(contrast)
    matches = [
        index
        for index, level in enumerate(config.contrast_levels)
        if np.isclose(contrast, level)
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Contrast {contrast:g} is not uniquely configured."
        )

    result = analysis.pca_results[region]
    source = (
        result.scores
        if config.pca_show_individual_trials
        else result.centroids
    )
    limits: dict[str, tuple[float, float]] = {}
    for column in PCA_PC_COLUMNS:
        values = source[column].to_numpy(dtype=float)
        low = float(np.nanmin(values))
        high = float(np.nanmax(values))
        span = high - low
        padding = 0.12 * span if span > 0 else 0.5
        limits[column] = (low - padding, high + padding)
    return (
        result,
        contrast,
        matches[0],
        _pca_contrast_color(contrast, config),
        limits,
    )


def _event_window_pca_sample_trial_ids(
    group: pd.DataFrame,
    choice_index: int,
    contrast_index: int,
    config: ProcessingConfig,
) -> np.ndarray:
    trial_ids = np.sort(group["trial_id"].unique().astype(int))
    maximum = config.pca_max_individual_trials_per_group
    if len(trial_ids) > maximum:
        rng = np.random.default_rng(
            config.random_state
            + 1000 * choice_index
            + contrast_index
        )
        trial_ids = np.sort(
            rng.choice(trial_ids, size=maximum, replace=False)
        )
    return trial_ids


def _event_window_pca_legend_handles() -> list[Line2D]:
    choice_handles = [
        Line2D(
            [0],
            [0],
            color="#5F6B75",
            linestyle=PCA_CHOICE_LINESTYLES["Right"],
            linewidth=2.5,
            marker="o",
            markersize=5.8,
            markerfacecolor="#5F6B75",
            markeredgecolor="white",
            label="Right choice",
        ),
        Line2D(
            [0],
            [0],
            color="#5F6B75",
            linestyle=PCA_CHOICE_LINESTYLES["Left"],
            linewidth=2.5,
            marker="o",
            markersize=5.8,
            markerfacecolor="white",
            markeredgecolor="#5F6B75",
            label="Left choice",
        ),
    ]
    phase_handles = [
        Line2D(
            [0],
            [0],
            linestyle="none",
            marker=marker,
            markersize=6.2,
            markerfacecolor="white",
            markeredgecolor="#66727D",
            markeredgewidth=1.0,
            label=label,
        )
        for marker, label in zip(PCA_PHASE_MARKERS, PCA_PHASE_LABELS)
    ]
    return [*choice_handles, *phase_handles]


def plot_event_window_pca_2d(
    analysis: SessionAnalysis,
    region: str,
    contrast: float,
    config: ProcessingConfig,
) -> tuple[plt.Figure, Any]:
    (
        result,
        contrast,
        contrast_index,
        color,
        limits,
    ) = _event_window_pca_plot_inputs(
        analysis,
        region,
        contrast,
        config,
    )
    variance = result.explained_variance_ratio * 100

    with plt.rc_context(_publication_rc()):
        fig, axis = plt.subplots(figsize=(7.8, 6.5))
        plotted_choices = 0
        for choice_index, choice in enumerate(("Right", "Left")):
            group = result.scores.loc[
                result.scores["choice"].eq(choice)
                & np.isclose(result.scores["contrast"], contrast)
            ]
            if config.pca_show_individual_trials:
                trial_ids = _event_window_pca_sample_trial_ids(
                    group,
                    choice_index,
                    contrast_index,
                    config,
                )
                for trial_id in trial_ids:
                    trial = group.loc[
                        group["trial_id"].eq(trial_id)
                    ].sort_values("phase_index")
                    _, trial_display = _trajectory_display_curve(
                        trial.loc[:, ["PC1", "PC2"]].to_numpy(
                            dtype=float
                        ),
                        smooth=True,
                        samples_per_segment=32,
                    )
                    line, = axis.plot(
                        trial_display[:, 0],
                        trial_display[:, 1],
                        color=color,
                        linestyle=PCA_CHOICE_LINESTYLES[choice],
                        linewidth=0.65,
                        alpha=config.pca_individual_alpha,
                        zorder=1,
                    )
                    line.set_gid("event-window-individual-trial:2d")
                    _style_trajectory_line(line, choice)

            trajectory = result.centroids.loc[
                np.isclose(result.centroids["contrast"], contrast)
                & result.centroids["choice"].eq(choice)
            ].sort_values("phase_index")
            if trajectory.empty:
                continue
            plotted_choices += 1
            if len(trajectory) != len(PCA_PHASES):
                raise ValueError(
                    f"{region} / {contrast:g} / {choice}: expected "
                    f"{len(PCA_PHASES)} event-window PCA states, "
                    f"received {len(trajectory)}."
                )

            coordinates = trajectory.loc[
                :, ["PC1", "PC2"]
            ].to_numpy(dtype=float)
            display_parameter, display_coordinates = (
                _trajectory_display_curve(
                    coordinates,
                    smooth=True,
                    samples_per_segment=48,
                )
            )
            line, = axis.plot(
                display_coordinates[:, 0],
                display_coordinates[:, 1],
                color=color,
                linestyle=PCA_CHOICE_LINESTYLES[choice],
                linewidth=3.05,
                alpha=1.0,
                zorder=5,
            )
            line.set_gid(
                f"event-window-centroid:2d:{contrast:g}:"
                f"{choice.lower()}"
            )
            _style_trajectory_line(line, choice)
            line.set_path_effects(
                [
                    path_effects.Stroke(
                        linewidth=5.9,
                        foreground="white",
                        alpha=0.96,
                    ),
                    path_effects.Normal(),
                ]
            )

            for phase_index in range(len(coordinates) - 1):
                arrow_segment = _trajectory_arrow_segment(
                    display_parameter,
                    display_coordinates,
                    float(phase_index),
                    float(phase_index + 1),
                )
                if arrow_segment is None:
                    continue
                _draw_pca_direction_arrow_2d(
                    axis,
                    arrow_segment[0],
                    arrow_segment[1],
                    color,
                    (
                        f"event-window-direction-arrow:2d:"
                        f"{contrast:g}:{choice.lower()}:"
                        f"{phase_index}"
                    ),
                )

            for marker, (_, point) in zip(
                PCA_PHASE_MARKERS,
                trajectory.iterrows(),
            ):
                is_right = choice == "Right"
                collection = axis.scatter(
                    point["PC1"],
                    point["PC2"],
                    s=64,
                    marker=marker,
                    facecolor=color if is_right else "white",
                    edgecolor="white" if is_right else color,
                    linewidth=1.1 if is_right else 1.7,
                    zorder=8,
                )
                collection.set_gid(
                    f"event-window-centroid-point:2d:{contrast:g}:"
                    f"{choice.lower()}:{point['phase']}"
                )

        if plotted_choices == 0:
            raise ValueError(
                f"No event-window PCA centroid is available for "
                f"{region} / {contrast:g}."
            )

        axis.set_xlabel(f"PC1 ({variance[0]:.1f}%)", labelpad=8)
        axis.set_ylabel(f"PC2 ({variance[1]:.1f}%)", labelpad=8)
        axis.set_xlim(*limits["PC1"])
        axis.set_ylim(*limits["PC2"])
        axis.set_aspect("equal", adjustable="box")
        axis.grid(True, color="#E7EBEF", linewidth=0.7, alpha=0.9)
        axis.set_axisbelow(True)
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)
        fig.legend(
            handles=_event_window_pca_legend_handles(),
            frameon=False,
            loc="lower center",
            bbox_to_anchor=(0.5, 0.018),
            ncol=6,
            columnspacing=1.05,
            handlelength=1.6,
            fontsize=7.8,
        )
        fig.suptitle(
            f"{region} event-window PCA — 4 neural states — "
            f"{contrast * 100:g}% contrast (2D)",
            x=0.075,
            y=0.975,
            ha="left",
            fontsize=12,
            fontweight="semibold",
        )
        fig.subplots_adjust(
            left=0.14,
            right=0.97,
            bottom=0.14,
            top=0.90,
        )
        return fig, axis


def plot_event_window_pca_3d(
    analysis: SessionAnalysis,
    region: str,
    contrast: float,
    config: ProcessingConfig,
) -> tuple[plt.Figure, Any]:
    (
        result,
        contrast,
        contrast_index,
        color,
        coordinate_limits,
    ) = _event_window_pca_plot_inputs(
        analysis,
        region,
        contrast,
        config,
    )
    scores = result.scores
    centroids = result.centroids
    variance = result.explained_variance_ratio * 100

    with plt.rc_context(_publication_rc()):
        fig = plt.figure(figsize=(8.8, 7.1))
        axis = fig.add_subplot(111, projection="3d")
        axis.set_proj_type("ortho")

        plotted_choices = 0
        for choice_index, choice in enumerate(("Right", "Left")):
            group = scores.loc[
                scores["choice"].eq(choice)
                & np.isclose(scores["contrast"], contrast)
            ]
            if config.pca_show_individual_trials:
                trial_ids = _event_window_pca_sample_trial_ids(
                    group,
                    choice_index,
                    contrast_index,
                    config,
                )
                for trial_id in trial_ids:
                    trial = group.loc[
                        group["trial_id"].eq(trial_id)
                    ].sort_values("phase_index")
                    _, trial_display = _trajectory_display_curve(
                        trial.loc[:, PCA_PC_COLUMNS].to_numpy(
                            dtype=float
                        ),
                        smooth=True,
                        samples_per_segment=32,
                    )
                    line, = axis.plot(
                        trial_display[:, 0],
                        trial_display[:, 1],
                        trial_display[:, 2],
                        color=color,
                        linestyle=PCA_CHOICE_LINESTYLES[choice],
                        linewidth=0.65,
                        alpha=config.pca_individual_alpha,
                        zorder=1,
                    )
                    line.set_gid("event-window-individual-trial:3d")
                    _style_trajectory_line(line, choice)

            trajectory = centroids.loc[
                np.isclose(centroids["contrast"], contrast)
                & centroids["choice"].eq(choice)
            ].sort_values("phase_index")
            if trajectory.empty:
                continue
            plotted_choices += 1
            if len(trajectory) != len(PCA_PHASES):
                raise ValueError(
                    f"{region} / {contrast:g} / {choice}: expected "
                    f"{len(PCA_PHASES)} PCA phases, received "
                    f"{len(trajectory)}."
                )

            coordinates = trajectory.loc[
                :, PCA_PC_COLUMNS
            ].to_numpy(dtype=float)
            display_parameter, display_coordinates = (
                _trajectory_display_curve(
                    coordinates,
                    smooth=True,
                    samples_per_segment=48,
                )
            )
            line, = axis.plot(
                display_coordinates[:, 0],
                display_coordinates[:, 1],
                display_coordinates[:, 2],
                color=color,
                linestyle=PCA_CHOICE_LINESTYLES[choice],
                linewidth=3.15,
                alpha=1.0,
                zorder=5,
            )
            line.set_gid(
                f"event-window-centroid:3d:{contrast:g}:"
                f"{choice.lower()}"
            )
            _style_trajectory_line(line, choice)
            line.set_path_effects(
                [
                    path_effects.Stroke(
                        linewidth=6.2,
                        foreground="white",
                        alpha=0.96,
                    ),
                    path_effects.Normal(),
                ]
            )

            for phase_index in range(len(coordinates) - 1):
                arrow_segment = _trajectory_arrow_segment(
                    display_parameter,
                    display_coordinates,
                    float(phase_index),
                    float(phase_index + 1),
                )
                if arrow_segment is None:
                    continue
                _draw_pca_direction_arrow(
                    axis,
                    arrow_segment[0],
                    arrow_segment[1],
                    color,
                    (
                        f"event-window-direction-arrow:3d:"
                        f"{contrast:g}:{choice.lower()}:"
                        f"{phase_index}"
                    ),
                )

            for marker, (_, point) in zip(
                PCA_PHASE_MARKERS,
                trajectory.iterrows(),
            ):
                is_right = choice == "Right"
                collection = axis.scatter(
                    point["PC1"],
                    point["PC2"],
                    point["PC3"],
                    s=66,
                    marker=marker,
                    facecolor=color if is_right else "white",
                    edgecolor="white" if is_right else color,
                    linewidth=1.15 if is_right else 1.75,
                    alpha=1.0,
                    depthshade=False,
                    zorder=8,
                )
                collection.set_gid(
                    f"event-window-centroid-point:3d:{contrast:g}:"
                    f"{choice.lower()}:{point['phase']}"
                )

        if plotted_choices == 0:
            raise ValueError(
                f"No event-window PCA centroid is available for {region} / "
                f"{contrast:g}."
            )

        axis.set_xlabel(f"PC1 ({variance[0]:.1f}%)", labelpad=7)
        axis.set_ylabel(f"PC2 ({variance[1]:.1f}%)", labelpad=7)
        axis.set_zlabel(f"PC3 ({variance[2]:.1f}%)", labelpad=7)
        axis.set_xlim(*coordinate_limits["PC1"])
        axis.set_ylim(*coordinate_limits["PC2"])
        axis.set_zlim(*coordinate_limits["PC3"])
        axis.view_init(
            elev=config.pca_view_elev,
            azim=config.pca_view_azim,
        )
        axis.set_box_aspect((1.0, 1.0, 0.88))
        axis.tick_params(labelsize=8.0, pad=1.5)
        axis.grid(False)

        for dimension in (axis.xaxis, axis.yaxis, axis.zaxis):
            dimension.pane.set_facecolor((1.0, 1.0, 1.0, 0.0))
            dimension.pane.set_edgecolor("#D7DEE4")
            dimension.line.set_color("#AEB8C1")

        fig.legend(
            handles=_event_window_pca_legend_handles(),
            frameon=False,
            loc="lower center",
            bbox_to_anchor=(0.5, 0.018),
            ncol=6,
            columnspacing=1.05,
            handlelength=1.6,
            fontsize=7.8,
        )
        fig.suptitle(
            f"{region} event-window PCA — 4 neural states — "
            f"{contrast * 100:g}% contrast (3D)",
            x=0.055,
            y=0.975,
            ha="left",
            fontsize=12,
            fontweight="semibold",
        )
        fig.subplots_adjust(
            left=0.015,
            right=0.985,
            bottom=0.13,
            top=0.91,
        )
        return fig, axis


def plot_pca_trial_trajectories(
    analysis: SessionAnalysis,
    region: str,
    contrast: float,
    config: ProcessingConfig,
) -> tuple[plt.Figure, Any]:
    '''Backward-compatible alias for the event-window 3D view.'''
    return plot_event_window_pca_3d(
        analysis,
        region,
        contrast,
        config,
    )


## 10. Continuous time-resolved PCA trajectories

This is an additional analysis; the four-state PCA above is unchanged. For every valid
trial, Gaussian-smoothed population firing rates are sampled from 200 ms before stimulus
onset, through a linearly normalized stimulus-to-action interval, to 200 ms after action.
The normalization gives every trial the same number of time points without reversing or
duplicating overlapping post-stimulus and pre-action windows.

One three-component scaler/PCA basis is fitted per `session × region` across all valid
trials, time points, contrasts, and choices. The same transformed coordinates feed two
separate contrast-specific figures: a clear `PC1–PC2` view and a full 3D view. Both show
Left and Right together, four event anchors, and three direction arrows per trajectory.


In [ ]:
CONTINUOUS_PCA_STAGES = (
    "Pre-stimulus",
    "Stimulus-to-action",
    "Post-action",
)
CONTINUOUS_PCA_ANCHOR_MARKERS = {
    "Start": "o",
    "Stimulus": "^",
    "Action": "s",
    "End": "D",
}


def continuous_pca_timepoints(
    config: ProcessingConfig,
) -> pd.DataFrame:
    step = config.continuous_pca_sample_step_s
    n_pre = int(round(config.continuous_pca_pre_stim_s / step))
    n_post = int(round(config.continuous_pca_post_action_s / step))
    n_decision = config.continuous_pca_decision_points

    pre_offsets = (
        -config.continuous_pca_pre_stim_s
        + np.arange(n_pre, dtype=float) * step
    )
    decision_progress = np.linspace(0.0, 1.0, n_decision)
    post_offsets = np.arange(1, n_post + 1, dtype=float) * step
    n_timepoints = n_pre + n_decision + n_post

    stage = np.concatenate(
        [
            np.repeat(CONTINUOUS_PCA_STAGES[0], n_pre),
            np.repeat(CONTINUOUS_PCA_STAGES[1], n_decision),
            np.repeat(CONTINUOUS_PCA_STAGES[2], n_post),
        ]
    )
    stage_index = np.concatenate(
        [
            np.arange(n_pre, dtype=int),
            np.arange(n_decision, dtype=int),
            np.arange(n_post, dtype=int),
        ]
    )
    relative_time_s = np.concatenate(
        [
            pre_offsets,
            np.full(n_decision, np.nan),
            post_offsets,
        ]
    )
    normalized_decision_time = np.concatenate(
        [
            np.full(n_pre, np.nan),
            decision_progress,
            np.full(n_post, np.nan),
        ]
    )
    anchors = np.repeat("", n_timepoints).astype(object)
    anchor_labels = np.repeat("", n_timepoints).astype(object)
    anchors[0] = "Start"
    anchor_labels[0] = (
        f"−{config.continuous_pca_pre_stim_s * 1000:g} ms"
    )
    stimulus_index = n_pre
    action_index = n_pre + n_decision - 1
    anchors[stimulus_index] = "Stimulus"
    anchor_labels[stimulus_index] = "Stimulus"
    anchors[action_index] = "Action"
    anchor_labels[action_index] = "Action"
    anchors[-1] = "End"
    anchor_labels[-1] = (
        f"+{config.continuous_pca_post_action_s * 1000:g} ms"
    )

    return pd.DataFrame(
        {
            "time_index": np.arange(n_timepoints, dtype=int),
            "stage": stage,
            "stage_index": stage_index,
            "relative_time_s": relative_time_s,
            "normalized_decision_time": normalized_decision_time,
            "anchor": anchors,
            "anchor_label": anchor_labels,
        }
    )


def continuous_pca_trial_data(
    context: dict[str, np.ndarray],
    config: ProcessingConfig,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    trial_ids, _, contrasts = decoder_trial_data(context)
    stimulus = context["stimOn_times"][trial_ids]
    action = context["firstMovement_times"][trial_ids]
    latencies = action - stimulus
    low, high = config.movement_latency_s
    configured_contrast = np.any(
        np.isclose(
            contrasts[:, None],
            np.asarray(config.contrast_levels, dtype=float)[None, :],
        ),
        axis=1,
    )
    keep = (
        np.isfinite(contrasts)
        & np.isfinite(latencies)
        & (latencies >= low)
        & (latencies <= high)
        & configured_contrast
    )
    trial_ids = trial_ids[keep]
    choice_values = context["choice"][trial_ids].astype(int)
    contrasts = contrasts[keep]
    latencies = latencies[keep]
    if len(trial_ids) < 1:
        raise ValueError(
            "Continuous PCA has no valid trials after latency and contrast checks."
        )
    return trial_ids, choice_values, contrasts, latencies


def continuous_pca_sample_times(
    context: dict[str, np.ndarray],
    trial_ids: np.ndarray,
    timepoints: pd.DataFrame,
) -> np.ndarray:
    trial_ids = np.asarray(trial_ids, dtype=int)
    stimulus = context["stimOn_times"][trial_ids]
    action = context["firstMovement_times"][trial_ids]
    if np.any(action <= stimulus):
        raise ValueError(
            "Continuous PCA requires action after stimulus for every trial."
        )

    sample_times = np.empty(
        (len(trial_ids), len(timepoints)),
        dtype=float,
    )
    pre_mask = timepoints["stage"].eq(CONTINUOUS_PCA_STAGES[0])
    decision_mask = timepoints["stage"].eq(
        CONTINUOUS_PCA_STAGES[1]
    )
    post_mask = timepoints["stage"].eq(CONTINUOUS_PCA_STAGES[2])
    sample_times[:, pre_mask] = (
        stimulus[:, None]
        + timepoints.loc[
            pre_mask, "relative_time_s"
        ].to_numpy(dtype=float)[None, :]
    )
    sample_times[:, decision_mask] = (
        stimulus[:, None]
        + (action - stimulus)[:, None]
        * timepoints.loc[
            decision_mask,
            "normalized_decision_time",
        ].to_numpy(dtype=float)[None, :]
    )
    sample_times[:, post_mask] = (
        action[:, None]
        + timepoints.loc[
            post_mask, "relative_time_s"
        ].to_numpy(dtype=float)[None, :]
    )
    if not np.all(np.isfinite(sample_times)):
        raise ValueError("Continuous-PCA sample times must be finite.")
    if np.any(np.diff(sample_times, axis=1) <= 0):
        raise ValueError(
            "Continuous-PCA aligned time must increase strictly."
        )
    return sample_times


def estimate_gaussian_firing_rates(
    spike_times: np.ndarray,
    spike_units: np.ndarray,
    unit_ids: np.ndarray,
    sample_times: np.ndarray,
    sigma_s: float,
    truncate_sigma: float = 4.0,
) -> np.ndarray:
    spike_times = np.asarray(spike_times, dtype=float)
    spike_units = np.asarray(spike_units, dtype=int)
    unit_ids = np.asarray(unit_ids, dtype=int)
    sample_times = np.asarray(sample_times, dtype=float)
    if sigma_s <= 0 or truncate_sigma <= 0:
        raise ValueError("Gaussian firing-rate parameters must be positive.")
    if sample_times.ndim != 2:
        raise ValueError("sample_times must be trial × timepoint.")
    if not len(unit_ids):
        raise ValueError("unit_ids cannot be empty.")
    if len(spike_times) != len(spike_units):
        raise ValueError("spike_times and spike_units must align.")
    if not len(spike_times):
        raise ValueError("Continuous PCA requires at least one spike.")
    if np.any(np.diff(spike_times) < 0):
        raise ValueError("spike_times must be sorted.")

    lookup_size = int(max(spike_units.max(), unit_ids.max())) + 1
    lookup = np.full(lookup_size, -1, dtype=int)
    lookup[unit_ids] = np.arange(len(unit_ids))
    output = np.zeros(
        (*sample_times.shape, len(unit_ids)),
        dtype=float,
    )
    radius = truncate_sigma * sigma_s
    normalization = sigma_s * np.sqrt(2.0 * np.pi)

    for trial_index, trial_times in enumerate(sample_times):
        for time_index, center in enumerate(trial_times):
            left = np.searchsorted(
                spike_times,
                center - radius,
                side="left",
            )
            right = np.searchsorted(
                spike_times,
                center + radius,
                side="right",
            )
            local_units = lookup[spike_units[left:right]]
            valid = local_units >= 0
            if not np.any(valid):
                continue
            offsets = spike_times[left:right][valid] - center
            weights = (
                np.exp(-0.5 * (offsets / sigma_s) ** 2)
                / normalization
            )
            output[trial_index, time_index] = np.bincount(
                local_units[valid],
                weights=weights,
                minlength=len(unit_ids),
            )
    return output


def build_continuous_pca_features(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    region: str,
    config: ProcessingConfig,
) -> tuple[
    np.ndarray,
    pd.DataFrame,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
]:
    trial_ids, choices, contrasts, latencies = (
        continuous_pca_trial_data(context, config)
    )
    unit_ids = neural.units.loc[
        neural.units["region"].eq(region), "unit_id"
    ].to_numpy(dtype=int)
    if len(unit_ids) < config.continuous_pca_n_components:
        raise ValueError(
            f"{region}: continuous PCA needs at least "
            f"{config.continuous_pca_n_components} units."
        )
    timepoints = continuous_pca_timepoints(config)
    sample_times = continuous_pca_sample_times(
        context,
        trial_ids,
        timepoints,
    )
    firing_rates = estimate_gaussian_firing_rates(
        neural.spike_times,
        neural.spike_units,
        unit_ids,
        sample_times,
        config.continuous_pca_smoothing_s,
    )
    return (
        firing_rates,
        timepoints,
        trial_ids,
        choices,
        contrasts,
        latencies,
    )


def fit_continuous_trial_pca(
    firing_rates: np.ndarray,
    timepoints: pd.DataFrame,
    trial_ids: np.ndarray,
    choice_values: np.ndarray,
    contrasts: np.ndarray,
    movement_latencies_s: np.ndarray,
    config: ProcessingConfig,
) -> ContinuousPCATrajectoryResult:
    firing_rates = np.asarray(firing_rates, dtype=float)
    trial_ids = np.asarray(trial_ids, dtype=int)
    choice_values = np.asarray(choice_values, dtype=int)
    contrasts = np.asarray(contrasts, dtype=float)
    movement_latencies_s = np.asarray(
        movement_latencies_s,
        dtype=float,
    )
    if firing_rates.ndim != 3:
        raise ValueError(
            "Continuous-PCA features must be trial × timepoint × unit."
        )
    n_trials, n_timepoints, n_units = firing_rates.shape
    if n_timepoints != len(timepoints):
        raise ValueError(
            "Continuous-PCA features and timepoint metadata do not align."
        )
    if not np.array_equal(
        timepoints["time_index"].to_numpy(dtype=int),
        np.arange(n_timepoints),
    ):
        raise ValueError("Continuous-PCA time_index must be contiguous.")
    if not (
        len(trial_ids)
        == len(choice_values)
        == len(contrasts)
        == len(movement_latencies_s)
        == n_trials
    ):
        raise ValueError("Continuous-PCA trial metadata must align.")
    if not np.all(np.isin(choice_values, [-1, 1])):
        raise ValueError(
            "Continuous-PCA choices must contain only -1 and +1."
        )
    if not np.all(np.isfinite(firing_rates)):
        raise ValueError(
            "Continuous-PCA firing rates contain non-finite values."
        )
    if not np.all(np.isfinite(contrasts)):
        raise ValueError("Continuous-PCA contrasts must be finite.")
    if min(n_trials * n_timepoints, n_units) < (
        config.continuous_pca_n_components
    ):
        raise ValueError(
            "Not enough samples or units for continuous PCA."
        )

    combined = firing_rates.reshape(n_trials * n_timepoints, n_units)
    scaler = StandardScaler()
    combined_scaled = scaler.fit_transform(combined)
    if np.linalg.matrix_rank(combined_scaled) < (
        config.continuous_pca_n_components
    ):
        raise ValueError(
            "Continuous standardized activity has rank below three."
        )
    model = PCA(
        n_components=config.continuous_pca_n_components,
        svd_solver="full",
    )
    model.fit(combined_scaled)
    # Every trial is transformed separately in the one shared basis.
    coordinates = np.vstack(
        [
            model.transform(scaler.transform(trial_rates))
            for trial_rates in firing_rates
        ]
    )

    scores = pd.DataFrame(coordinates, columns=PCA_PC_COLUMNS)
    scores.insert(
        0,
        "time_index",
        np.tile(
            timepoints["time_index"].to_numpy(dtype=int),
            n_trials,
        ),
    )
    scores.insert(0, "contrast", np.repeat(contrasts, n_timepoints))
    scores.insert(
        0,
        "choice",
        np.repeat(
            np.where(choice_values == 1, "Right", "Left"),
            n_timepoints,
        ),
    )
    scores.insert(
        0,
        "choice_value",
        np.repeat(choice_values, n_timepoints),
    )
    scores.insert(0, "trial_id", np.repeat(trial_ids, n_timepoints))
    tiled_timepoints = pd.concat(
        [timepoints] * n_trials,
        ignore_index=True,
    ).drop(columns="time_index")
    scores = pd.concat([scores, tiled_timepoints], axis=1)

    centroids = (
        scores.groupby(
            [
                "contrast",
                "choice",
                "choice_value",
                "time_index",
            ],
            as_index=False,
            sort=False,
        )
        .agg(
            n_trials=("trial_id", "nunique"),
            PC1=("PC1", "mean"),
            PC2=("PC2", "mean"),
            PC3=("PC3", "mean"),
        )
        .merge(
            timepoints,
            on="time_index",
            how="left",
            validate="many_to_one",
        )
        .sort_values(
            ["contrast", "choice", "time_index"],
            kind="stable",
        )
        .reset_index(drop=True)
    )
    return ContinuousPCATrajectoryResult(
        scaler=scaler,
        model=model,
        scores=scores,
        centroids=centroids,
        timepoints=timepoints.copy(),
        movement_latencies_s=movement_latencies_s.copy(),
        explained_variance_ratio=(
            model.explained_variance_ratio_.copy()
        ),
        n_trials=n_trials,
        n_units=n_units,
        n_timepoints=n_timepoints,
    )


def run_continuous_pca_analysis(
    neural: NeuralSessionData,
    context: dict[str, np.ndarray],
    mapped_regions: tuple[str, ...],
    config: ProcessingConfig,
) -> tuple[
    dict[str, ContinuousPCATrajectoryResult],
    pd.DataFrame,
    pd.DataFrame,
]:
    results: dict[str, ContinuousPCATrajectoryResult] = {}
    summary_rows: list[dict] = []
    errors: list[dict] = []

    for region in mapped_regions:
        try:
            (
                firing_rates,
                timepoints,
                trial_ids,
                choices,
                contrasts,
                latencies,
            ) = build_continuous_pca_features(
                neural,
                context,
                region,
                config,
            )
            result = fit_continuous_trial_pca(
                firing_rates,
                timepoints,
                trial_ids,
                choices,
                contrasts,
                latencies,
                config,
            )
            results[region] = result
            variance = result.explained_variance_ratio
            summary_rows.append(
                {
                    "eid": neural.eid,
                    "subject": neural.subject,
                    "region": region,
                    "n_trials": result.n_trials,
                    "n_units": result.n_units,
                    "n_timepoints": result.n_timepoints,
                    "median_movement_latency_ms": (
                        np.median(result.movement_latencies_s) * 1000
                    ),
                    "pc1_variance": variance[0],
                    "pc2_variance": variance[1],
                    "pc3_variance": variance[2],
                    "pc1_pc2_variance": variance[:2].sum(),
                    "pc1_pc2_pc3_variance": variance.sum(),
                }
            )
        except Exception as exc:
            errors.append(
                make_error(
                    "continuous_pca",
                    neural.eid,
                    region,
                    exc,
                )
            )
            if not config.continue_on_error:
                raise

    return results, pd.DataFrame(summary_rows), pd.DataFrame(errors)


def _continuous_pca_plot_inputs(
    analysis: SessionAnalysis,
    region: str,
    contrast: float,
    config: ProcessingConfig,
) -> tuple[
    ContinuousPCATrajectoryResult,
    float,
    int,
    str,
    dict[str, tuple[float, float]],
]:
    if region not in analysis.continuous_pca_results:
        raise ValueError(
            f"No continuous PCA result for region {region}."
        )
    contrast = float(contrast)
    matches = [
        index
        for index, level in enumerate(config.contrast_levels)
        if np.isclose(contrast, level)
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Contrast {contrast:g} is not uniquely configured."
        )
    result = analysis.continuous_pca_results[region]
    source = (
        result.scores
        if config.continuous_pca_show_individual_trials
        else result.centroids
    )
    limits: dict[str, tuple[float, float]] = {}
    for column in PCA_PC_COLUMNS:
        values = source[column].to_numpy(dtype=float)
        low = float(np.nanmin(values))
        high = float(np.nanmax(values))
        span = high - low
        padding = 0.12 * span if span > 0 else 0.5
        limits[column] = (low - padding, high + padding)
    return (
        result,
        contrast,
        matches[0],
        _pca_contrast_color(contrast, config),
        limits,
    )


def _continuous_pca_arrow_ranges(
    timepoints: pd.DataFrame,
) -> tuple[tuple[str, int, int], ...]:
    ranges: list[tuple[str, int, int]] = []
    for stage in CONTINUOUS_PCA_STAGES:
        indices = timepoints.loc[
            timepoints["stage"].eq(stage),
            "time_index",
        ].to_numpy(dtype=int)
        if len(indices) < 2:
            continue
        start = indices[0]
        end = indices[-1]
        ranges.append(
            (stage.lower().replace("-", "_"), int(start), int(end))
        )
    return tuple(ranges)


def _continuous_pca_sample_trial_ids(
    group: pd.DataFrame,
    choice_index: int,
    contrast_index: int,
    config: ProcessingConfig,
) -> np.ndarray:
    trial_ids = np.sort(group["trial_id"].unique().astype(int))
    maximum = config.continuous_pca_max_individual_trials_per_group
    if len(trial_ids) > maximum:
        rng = np.random.default_rng(
            config.random_state
            + 1000 * choice_index
            + contrast_index
        )
        trial_ids = np.sort(
            rng.choice(
                trial_ids,
                size=maximum,
                replace=False,
            )
        )
    return trial_ids


def _draw_continuous_pca_direction_arrow_2d(
    axis: Any,
    start: np.ndarray,
    end: np.ndarray,
    color: str,
    gid: str,
) -> Any | None:
    return _draw_pca_direction_arrow_2d(
        axis,
        start,
        end,
        color,
        gid,
        mutation_scale=13.5,
    )


def _continuous_pca_legend_handles(
    result: ContinuousPCATrajectoryResult,
) -> list[Line2D]:
    handles = [
        Line2D(
            [0],
            [0],
            color="#5F6B75",
            linestyle=PCA_CHOICE_LINESTYLES["Right"],
            linewidth=2.5,
            marker="o",
            markersize=5.8,
            markerfacecolor="#5F6B75",
            markeredgecolor="white",
            label="Right choice",
        ),
        Line2D(
            [0],
            [0],
            color="#5F6B75",
            linestyle=PCA_CHOICE_LINESTYLES["Left"],
            linewidth=2.5,
            marker="o",
            markersize=5.8,
            markerfacecolor="white",
            markeredgecolor="#5F6B75",
            label="Left choice",
        ),
    ]
    anchors = result.timepoints.loc[
        result.timepoints["anchor"].ne("")
    ]
    handles.extend(
        Line2D(
            [0],
            [0],
            linestyle="none",
            marker=CONTINUOUS_PCA_ANCHOR_MARKERS[row.anchor],
            markersize=6.2,
            markerfacecolor="white",
            markeredgecolor="#66727D",
            markeredgewidth=1.0,
            label=row.anchor_label,
        )
        for row in anchors.itertuples(index=False)
    )
    return handles


def plot_continuous_pca_2d(
    analysis: SessionAnalysis,
    region: str,
    contrast: float,
    config: ProcessingConfig,
) -> tuple[plt.Figure, Any]:
    (
        result,
        contrast,
        contrast_index,
        color,
        limits,
    ) = _continuous_pca_plot_inputs(
        analysis,
        region,
        contrast,
        config,
    )
    variance = result.explained_variance_ratio * 100
    arrow_ranges = _continuous_pca_arrow_ranges(result.timepoints)

    with plt.rc_context(_publication_rc()):
        fig, axis = plt.subplots(figsize=(7.8, 6.5))
        plotted_choices = 0
        for choice_index, choice in enumerate(("Right", "Left")):
            group = result.scores.loc[
                result.scores["choice"].eq(choice)
                & np.isclose(result.scores["contrast"], contrast)
            ]
            if config.continuous_pca_show_individual_trials:
                trial_ids = _continuous_pca_sample_trial_ids(
                    group,
                    choice_index,
                    contrast_index,
                    config,
                )
                for trial_id in trial_ids:
                    trial = group.loc[
                        group["trial_id"].eq(trial_id)
                    ].sort_values("time_index")
                    line, = axis.plot(
                        trial["PC1"],
                        trial["PC2"],
                        color=color,
                        linestyle=PCA_CHOICE_LINESTYLES[choice],
                        linewidth=0.65,
                        alpha=(
                            config.continuous_pca_individual_alpha
                        ),
                        zorder=1,
                    )
                    line.set_gid("continuous-individual-trial:2d")
                    _style_trajectory_line(line, choice)

            trajectory = result.centroids.loc[
                np.isclose(result.centroids["contrast"], contrast)
                & result.centroids["choice"].eq(choice)
            ].sort_values("time_index")
            if trajectory.empty:
                continue
            if len(trajectory) != result.n_timepoints:
                raise ValueError(
                    f"{region} / {contrast:g} / {choice}: expected "
                    f"{result.n_timepoints} continuous PCA points, "
                    f"received {len(trajectory)}."
                )
            plotted_choices += 1
            coordinates = trajectory.loc[
                :, ["PC1", "PC2"]
            ].to_numpy(dtype=float)
            display_parameter, display_coordinates = (
                _trajectory_display_curve(
                    coordinates,
                    smooth=True,
                    samples_per_segment=8,
                )
            )
            line, = axis.plot(
                display_coordinates[:, 0],
                display_coordinates[:, 1],
                color=color,
                linestyle=PCA_CHOICE_LINESTYLES[choice],
                linewidth=2.85,
                alpha=1.0,
                zorder=5,
            )
            line.set_gid(
                f"continuous-centroid:2d:{contrast:g}:"
                f"{choice.lower()}"
            )
            _style_trajectory_line(line, choice)
            line.set_path_effects(
                [
                    path_effects.Stroke(
                        linewidth=5.6,
                        foreground="white",
                        alpha=0.96,
                    ),
                    path_effects.Normal(),
                ]
            )
            for stage, start_index, end_index in arrow_ranges:
                arrow_segment = _trajectory_arrow_segment(
                    display_parameter,
                    display_coordinates,
                    float(start_index),
                    float(end_index),
                )
                if arrow_segment is None:
                    continue
                _draw_continuous_pca_direction_arrow_2d(
                    axis,
                    arrow_segment[0],
                    arrow_segment[1],
                    color,
                    (
                        f"continuous-direction-arrow:2d:{contrast:g}:"
                        f"{choice.lower()}:{stage}"
                    ),
                )

            for point in trajectory.loc[
                trajectory["anchor"].ne("")
            ].itertuples(index=False):
                is_right = choice == "Right"
                collection = axis.scatter(
                    point.PC1,
                    point.PC2,
                    s=64,
                    marker=(
                        CONTINUOUS_PCA_ANCHOR_MARKERS[point.anchor]
                    ),
                    facecolor=color if is_right else "white",
                    edgecolor="white" if is_right else color,
                    linewidth=1.1 if is_right else 1.7,
                    zorder=8,
                )
                collection.set_gid(
                    f"continuous-centroid-point:2d:{contrast:g}:"
                    f"{choice.lower()}:{point.anchor.lower()}"
                )

        if plotted_choices != 2:
            raise ValueError(
                f"Continuous PCA requires both choices for {region} / "
                f"{contrast:g}; received {plotted_choices}."
            )
        axis.set_xlabel(f"PC1 ({variance[0]:.1f}%)", labelpad=8)
        axis.set_ylabel(f"PC2 ({variance[1]:.1f}%)", labelpad=8)
        axis.set_xlim(*limits["PC1"])
        axis.set_ylim(*limits["PC2"])
        axis.set_aspect("equal", adjustable="box")
        axis.grid(
            True,
            color="#E7EBEF",
            linewidth=0.7,
            alpha=0.9,
        )
        axis.set_axisbelow(True)
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)
        fig.legend(
            handles=_continuous_pca_legend_handles(result),
            frameon=False,
            loc="lower center",
            bbox_to_anchor=(0.5, 0.018),
            ncol=6,
            columnspacing=1.05,
            handlelength=1.6,
            fontsize=7.8,
        )
        fig.suptitle(
            f"{region} continuous PCA — time-resolved — "
            f"{contrast * 100:g}% contrast (2D)",
            x=0.075,
            y=0.975,
            ha="left",
            fontsize=12,
            fontweight="semibold",
        )
        fig.subplots_adjust(
            left=0.14,
            right=0.97,
            bottom=0.14,
            top=0.90,
        )
        return fig, axis


def plot_continuous_pca_3d(
    analysis: SessionAnalysis,
    region: str,
    contrast: float,
    config: ProcessingConfig,
) -> tuple[plt.Figure, Any]:
    (
        result,
        contrast,
        contrast_index,
        color,
        limits,
    ) = _continuous_pca_plot_inputs(
        analysis,
        region,
        contrast,
        config,
    )
    variance = result.explained_variance_ratio * 100
    arrow_ranges = _continuous_pca_arrow_ranges(result.timepoints)

    with plt.rc_context(_publication_rc()):
        fig = plt.figure(figsize=(8.8, 7.1))
        axis = fig.add_subplot(111, projection="3d")
        axis.set_proj_type("ortho")
        plotted_choices = 0

        for choice_index, choice in enumerate(("Right", "Left")):
            group = result.scores.loc[
                result.scores["choice"].eq(choice)
                & np.isclose(result.scores["contrast"], contrast)
            ]
            if config.continuous_pca_show_individual_trials:
                trial_ids = _continuous_pca_sample_trial_ids(
                    group,
                    choice_index,
                    contrast_index,
                    config,
                )
                for trial_id in trial_ids:
                    trial = group.loc[
                        group["trial_id"].eq(trial_id)
                    ].sort_values("time_index")
                    line, = axis.plot(
                        trial["PC1"],
                        trial["PC2"],
                        trial["PC3"],
                        color=color,
                        linestyle=PCA_CHOICE_LINESTYLES[choice],
                        linewidth=0.65,
                        alpha=(
                            config.continuous_pca_individual_alpha
                        ),
                        zorder=1,
                    )
                    line.set_gid("continuous-individual-trial:3d")
                    _style_trajectory_line(line, choice)

            trajectory = result.centroids.loc[
                np.isclose(result.centroids["contrast"], contrast)
                & result.centroids["choice"].eq(choice)
            ].sort_values("time_index")
            if trajectory.empty:
                continue
            if len(trajectory) != result.n_timepoints:
                raise ValueError(
                    f"{region} / {contrast:g} / {choice}: expected "
                    f"{result.n_timepoints} continuous PCA points, "
                    f"received {len(trajectory)}."
                )
            plotted_choices += 1
            coordinates = trajectory.loc[
                :, PCA_PC_COLUMNS
            ].to_numpy(dtype=float)
            display_parameter, display_coordinates = (
                _trajectory_display_curve(
                    coordinates,
                    smooth=True,
                    samples_per_segment=8,
                )
            )
            line, = axis.plot(
                display_coordinates[:, 0],
                display_coordinates[:, 1],
                display_coordinates[:, 2],
                color=color,
                linestyle=PCA_CHOICE_LINESTYLES[choice],
                linewidth=3.0,
                alpha=1.0,
                zorder=5,
            )
            line.set_gid(
                f"continuous-centroid:3d:{contrast:g}:"
                f"{choice.lower()}"
            )
            _style_trajectory_line(line, choice)
            line.set_path_effects(
                [
                    path_effects.Stroke(
                        linewidth=5.9,
                        foreground="white",
                        alpha=0.96,
                    ),
                    path_effects.Normal(),
                ]
            )
            for stage, start_index, end_index in arrow_ranges:
                arrow_segment = _trajectory_arrow_segment(
                    display_parameter,
                    display_coordinates,
                    float(start_index),
                    float(end_index),
                )
                if arrow_segment is None:
                    continue
                _draw_pca_direction_arrow(
                    axis,
                    arrow_segment[0],
                    arrow_segment[1],
                    color,
                    (
                        f"continuous-direction-arrow:3d:{contrast:g}:"
                        f"{choice.lower()}:{stage}"
                    ),
                    mutation_scale=13.0,
                )

            for point in trajectory.loc[
                trajectory["anchor"].ne("")
            ].itertuples(index=False):
                is_right = choice == "Right"
                collection = axis.scatter(
                    point.PC1,
                    point.PC2,
                    point.PC3,
                    s=66,
                    marker=(
                        CONTINUOUS_PCA_ANCHOR_MARKERS[point.anchor]
                    ),
                    facecolor=color if is_right else "white",
                    edgecolor="white" if is_right else color,
                    linewidth=1.15 if is_right else 1.75,
                    depthshade=False,
                    zorder=8,
                )
                collection.set_gid(
                    f"continuous-centroid-point:3d:{contrast:g}:"
                    f"{choice.lower()}:{point.anchor.lower()}"
                )

        if plotted_choices != 2:
            raise ValueError(
                f"Continuous PCA requires both choices for {region} / "
                f"{contrast:g}; received {plotted_choices}."
            )
        axis.set_xlabel(f"PC1 ({variance[0]:.1f}%)", labelpad=7)
        axis.set_ylabel(f"PC2 ({variance[1]:.1f}%)", labelpad=7)
        axis.set_zlabel(f"PC3 ({variance[2]:.1f}%)", labelpad=7)
        axis.set_xlim(*limits["PC1"])
        axis.set_ylim(*limits["PC2"])
        axis.set_zlim(*limits["PC3"])
        axis.view_init(
            elev=config.pca_view_elev,
            azim=config.pca_view_azim,
        )
        axis.set_box_aspect((1.0, 1.0, 0.88))
        axis.tick_params(labelsize=8.0, pad=1.5)
        axis.grid(False)
        for dimension in (axis.xaxis, axis.yaxis, axis.zaxis):
            dimension.pane.set_facecolor((1.0, 1.0, 1.0, 0.0))
            dimension.pane.set_edgecolor("#D7DEE4")
            dimension.line.set_color("#AEB8C1")
        fig.legend(
            handles=_continuous_pca_legend_handles(result),
            frameon=False,
            loc="lower center",
            bbox_to_anchor=(0.5, 0.018),
            ncol=6,
            columnspacing=1.05,
            handlelength=1.6,
            fontsize=7.8,
        )
        fig.suptitle(
            f"{region} continuous PCA — time-resolved — "
            f"{contrast * 100:g}% contrast (3D)",
            x=0.055,
            y=0.975,
            ha="left",
            fontsize=12,
            fontweight="semibold",
        )
        fig.subplots_adjust(
            left=0.015,
            right=0.985,
            bottom=0.13,
            top=0.91,
        )
        return fig, axis


## 11. End-to-end session processing


In [ ]:
def analyze_one_session(
    one: ONE,
    eid: str,
    mapped_regions: tuple[str, ...],
    config: ProcessingConfig,
    atlas: AllenAtlas,
    brain_regions: BrainRegions,
) -> SessionAnalysis:
    trials = load_trials(one, eid)
    context = prepare_trial_context(trials)
    hesitation = load_hesitation_result(one, eid, context, config)
    context = attach_hesitation_to_context(context, hesitation)
    neural = load_neural_session(
        one, eid, mapped_regions, config, atlas, brain_regions
    )

    peth_results, condition_counts, peth_errors = run_peth_analysis(
        neural, context, mapped_regions, config
    )
    (
        hesitation_peth_results,
        hesitation_condition_counts,
        hesitation_peth_errors,
    ) = run_hesitation_peth_analysis(
        neural, context, mapped_regions, config
    )
    (
        hesitation_presence_peth_results,
        hesitation_presence_condition_counts,
        hesitation_presence_peth_errors,
    ) = run_hesitation_presence_peth_analysis(
        neural, context, mapped_regions, config
    )
    (
        no_hesitation_vs_q4_peth_results,
        no_hesitation_vs_q4_condition_counts,
        no_hesitation_vs_q4_peth_errors,
    ) = run_no_hesitation_vs_q4_peth_analysis(
        neural, context, mapped_regions, config
    )
    (
        decoder_results,
        decoder_summary,
        decoder_errors,
        decoder_feature_cache,
    ) = run_decoder_analysis(neural, context, mapped_regions, config)
    pca_results, pca_summary, pca_errors = run_pca_analysis(
        neural,
        context,
        mapped_regions,
        config,
        feature_cache=decoder_feature_cache,
    )
    (
        continuous_pca_results,
        continuous_pca_summary,
        continuous_pca_errors,
    ) = run_continuous_pca_analysis(
        neural,
        context,
        mapped_regions,
        config,
    )
    unit_summary = (
        neural.units.groupby(["region", "hemisphere", "probe_name"], as_index=False)
        .size()
        .rename(columns={"size": "n_good_units"})
    )

    return SessionAnalysis(
        eid=eid,
        subject=neural.subject,
        unit_summary=unit_summary,
        region_probes=neural.region_probes,
        peth_results=peth_results,
        peth_condition_counts=condition_counts,
        peth_errors=peth_errors,
        hesitation=hesitation,
        hesitation_peth_results=hesitation_peth_results,
        hesitation_peth_condition_counts=hesitation_condition_counts,
        hesitation_peth_errors=hesitation_peth_errors,
        hesitation_presence_peth_results=(
            hesitation_presence_peth_results
        ),
        hesitation_presence_peth_condition_counts=(
            hesitation_presence_condition_counts
        ),
        hesitation_presence_peth_errors=(
            hesitation_presence_peth_errors
        ),
        no_hesitation_vs_q4_peth_results=(
            no_hesitation_vs_q4_peth_results
        ),
        no_hesitation_vs_q4_peth_condition_counts=(
            no_hesitation_vs_q4_condition_counts
        ),
        no_hesitation_vs_q4_peth_errors=(
            no_hesitation_vs_q4_peth_errors
        ),
        decoder_results=decoder_results,
        decoder_summary=decoder_summary,
        decoder_errors=decoder_errors,
        pca_results=pca_results,
        pca_summary=pca_summary,
        pca_errors=pca_errors,
        continuous_pca_results=continuous_pca_results,
        continuous_pca_summary=continuous_pca_summary,
        continuous_pca_errors=continuous_pca_errors,
        probe_errors=neural.probe_errors,
    )


def analyze_sessions(
    one: ONE,
    config: ProcessingConfig,
) -> tuple[dict[str, SessionAnalysis], pd.DataFrame, tuple[str, ...]]:
    validate_processing_config(config)
    atlas = AllenAtlas()
    brain_regions = BrainRegions()
    mapped_regions = map_requested_regions(
        brain_regions, config.regions, config.atlas_mapping
    )

    results: dict[str, SessionAnalysis] = {}
    errors: list[dict] = []
    for position, eid in enumerate(config.session_ids, start=1):
        print(f"[{position}/{len(config.session_ids)}] Processing {eid}")
        try:
            results[eid] = analyze_one_session(
                one, eid, mapped_regions, config, atlas, brain_regions
            )
            print(f"  completed: mouse {results[eid].subject}")
        except Exception as exc:
            errors.append(make_error("session", eid, None, exc))
            print(f"  failed: {type(exc).__name__}: {exc}")
            if not config.continue_on_error:
                raise
        finally:
            gc.collect()

    return results, pd.DataFrame(errors), mapped_regions


## 12. Run all requested sessions


In [ ]:
analysis_results, session_errors, MAPPED_REGIONS = analyze_sessions(one, CONFIG)

print("Mapped regions:", MAPPED_REGIONS)
print("Successful sessions:", len(analysis_results))
print("Failed sessions:", len(session_errors))
if not session_errors.empty:
    display(session_errors)


## 13. Inspect tables and draw figures

Every `session × region` pair produces eight non-PCA figures. Each contrast additionally
receives 2D and 3D event-window PCA figures plus 2D and 3D continuous PCA figures. Left
and Right remain together, and figures from the same PCA fit share identical PC limits.
Outputs are separated under `peth/`, `svm/`, `pca_event_windows/`, and
`pca_continuous/`; session-level behavior QC is stored under `session_qc/`. Every figure
is saved only as a 300-dpi PNG.


In [ ]:
decoder_summary_df = pd.concat(
    [result.decoder_summary for result in analysis_results.values()],
    ignore_index=True,
) if analysis_results else pd.DataFrame()

if not decoder_summary_df.empty:
    display(decoder_summary_df)

event_window_pca_summary_df = pd.concat(
    [result.pca_summary for result in analysis_results.values()],
    ignore_index=True,
) if analysis_results else pd.DataFrame()

if not event_window_pca_summary_df.empty:
    display(event_window_pca_summary_df)

continuous_pca_summary_df = pd.concat(
    [
        result.continuous_pca_summary
        for result in analysis_results.values()
    ],
    ignore_index=True,
) if analysis_results else pd.DataFrame()

if not continuous_pca_summary_df.empty:
    display(continuous_pca_summary_df)

figure_rows = []

for eid, analysis in analysis_results.items():
    print("=" * 80)
    print("Mouse:", analysis.subject)
    print("EID:", eid)
    display(analysis.region_probes)
    display(analysis.unit_summary)
    display(analysis.hesitation.state_summary)
    display(analysis.hesitation.summary)
    display(
        analysis.peth_condition_counts.pivot_table(
            index=["region", "hemisphere", "contrast"],
            columns="probability_left",
            values="n_trials",
            aggfunc="sum",
        )
    )
    display(
        analysis.hesitation_peth_condition_counts.pivot_table(
            index=["region", "hemisphere", "hesitation_quartile"],
            columns="probability_left",
            values="n_trials",
            aggfunc="sum",
        )
    )
    display(
        analysis.hesitation_peth_condition_counts[
            [
                "region",
                "probability_left",
                "used_for_balancing",
                "prior_status",
            ]
        ].drop_duplicates(ignore_index=True)
    )
    display(
        analysis.hesitation_presence_peth_condition_counts.pivot_table(
            index=["region", "hemisphere", "has_hesitation"],
            columns="probability_left",
            values="n_trials",
            aggfunc="sum",
        )
    )
    display(
        analysis.hesitation_presence_peth_condition_counts[
            [
                "region",
                "probability_left",
                "used_for_balancing",
                "prior_status",
            ]
        ].drop_duplicates(ignore_index=True)
    )

    display(
        analysis.no_hesitation_vs_q4_peth_condition_counts.pivot_table(
            index=[
                "region",
                "hemisphere",
                "no_hesitation_vs_q4",
            ],
            columns="probability_left",
            values="n_trials",
            aggfunc="sum",
        )
    )
    display(
        analysis.no_hesitation_vs_q4_peth_condition_counts[
            [
                "region",
                "probability_left",
                "used_for_balancing",
                "prior_status",
            ]
        ].drop_duplicates(ignore_index=True)
    )

    hesitation_qc_figure, _ = plot_hesitation_score_quartiles(
        analysis, CONFIG
    )
    hesitation_qc_paths = save_session_figure(
        hesitation_qc_figure,
        analysis,
        "hesitation_score_quartiles",
        CONFIG,
        analysis_path=("session_qc",),
    )
    figure_rows.extend(
        {
            "eid": eid,
            "region": "Session behavior",
            "analysis": "session_qc",
            "plot": "Hesitation QC",
            "phase": "Raw score + quartiles",
            "file": str(path),
        }
        for path in hesitation_qc_paths
    )
    plt.show()
    plt.close(hesitation_qc_figure)

    for region in MAPPED_REGIONS:
        if region in analysis.peth_results:
            peth_figure, _ = plot_event_aligned_population_activity(
                analysis, region, CONFIG
            )
            peth_paths = save_publication_figure(
                peth_figure,
                analysis,
                region,
                "peth_stimulus_and_action",
                CONFIG,
                analysis_path=("peth", "contrast"),
            )
            figure_rows.extend(
                {
                    "eid": eid,
                    "region": region,
                    "analysis": "peth/contrast",
                    "plot": "PETH",
                    "phase": "Stimulus + action",
                    "file": str(path),
                }
                for path in peth_paths
            )
            plt.show()
            plt.close(peth_figure)

        if region in analysis.hesitation_peth_results:
            hesitation_peth_figure, _ = (
                plot_event_aligned_activity_by_hesitation(
                    analysis, region, CONFIG
                )
            )
            hesitation_peth_paths = save_publication_figure(
                hesitation_peth_figure,
                analysis,
                region,
                "peth_hesitation_quartiles",
                CONFIG,
                analysis_path=("peth", "hesitation_quartiles"),
            )
            figure_rows.extend(
                {
                    "eid": eid,
                    "region": region,
                    "analysis": "peth/hesitation_quartiles",
                    "plot": "PETH — hesitation",
                    "phase": "Stimulus + action",
                    "file": str(path),
                }
                for path in hesitation_peth_paths
            )
            plt.show()
            plt.close(hesitation_peth_figure)

        if region in analysis.hesitation_presence_peth_results:
            hesitation_presence_peth_figure, _ = (
                plot_event_aligned_activity_by_hesitation_presence(
                    analysis, region, CONFIG
                )
            )
            hesitation_presence_peth_paths = save_publication_figure(
                hesitation_presence_peth_figure,
                analysis,
                region,
                "peth_hesitation_presence",
                CONFIG,
                analysis_path=("peth", "hesitation_presence"),
            )
            figure_rows.extend(
                {
                    "eid": eid,
                    "region": region,
                    "analysis": "peth/hesitation_presence",
                    "plot": "PETH — hesitation presence",
                    "phase": "Stimulus + action",
                    "file": str(path),
                }
                for path in hesitation_presence_peth_paths
            )
            plt.show()
            plt.close(hesitation_presence_peth_figure)

        if region in analysis.no_hesitation_vs_q4_peth_results:
            no_hesitation_vs_q4_peth_figure, _ = (
                plot_event_aligned_activity_no_hesitation_vs_q4(
                    analysis, region, CONFIG
                )
            )
            no_hesitation_vs_q4_peth_paths = save_publication_figure(
                no_hesitation_vs_q4_peth_figure,
                analysis,
                region,
                "peth_no_hesitation_vs_q4",
                CONFIG,
                analysis_path=("peth", "no_hesitation_vs_q4"),
            )
            figure_rows.extend(
                {
                    "eid": eid,
                    "region": region,
                    "analysis": "peth/no_hesitation_vs_q4",
                    "plot": "PETH — no hesitation vs Q4",
                    "phase": "Stimulus + action",
                    "file": str(path),
                }
                for path in no_hesitation_vs_q4_peth_paths
            )
            plt.show()
            plt.close(no_hesitation_vs_q4_peth_figure)

        if region in analysis.decoder_results:
            decoder_figure, _ = plot_decoder_accuracy_by_contrast(
                analysis, region, CONFIG
            )
            decoder_paths = save_publication_figure(
                decoder_figure,
                analysis,
                region,
                "svm_all_phases",
                CONFIG,
                analysis_path=("svm", "contrast"),
            )
            figure_rows.extend(
                {
                    "eid": eid,
                    "region": region,
                    "analysis": "svm/contrast",
                    "plot": "SVM",
                    "phase": "All phases",
                    "file": str(path),
                }
                for path in decoder_paths
            )
            plt.show()
            plt.close(decoder_figure)

            hesitation_decoder_figure, _ = (
                plot_decoder_accuracy_by_hesitation(
                    analysis, region, CONFIG
                )
            )
            hesitation_decoder_paths = save_publication_figure(
                hesitation_decoder_figure,
                analysis,
                region,
                "svm_hesitation_quartiles",
                CONFIG,
                analysis_path=("svm", "hesitation_quartiles"),
            )
            figure_rows.extend(
                {
                    "eid": eid,
                    "region": region,
                    "analysis": "svm/hesitation_quartiles",
                    "plot": "SVM — hesitation",
                    "phase": "All phases",
                    "file": str(path),
                }
                for path in hesitation_decoder_paths
            )
            plt.show()
            plt.close(hesitation_decoder_figure)

            hesitation_presence_decoder_figure, _ = (
                plot_decoder_accuracy_by_hesitation_presence(
                    analysis, region, CONFIG
                )
            )
            hesitation_presence_decoder_paths = save_publication_figure(
                hesitation_presence_decoder_figure,
                analysis,
                region,
                "svm_hesitation_presence",
                CONFIG,
                analysis_path=("svm", "hesitation_presence"),
            )
            figure_rows.extend(
                {
                    "eid": eid,
                    "region": region,
                    "analysis": "svm/hesitation_presence",
                    "plot": "SVM — hesitation presence",
                    "phase": "All phases",
                    "file": str(path),
                }
                for path in hesitation_presence_decoder_paths
            )
            plt.show()
            plt.close(hesitation_presence_decoder_figure)

            no_hesitation_vs_q4_decoder_figure, _ = (
                plot_decoder_accuracy_no_hesitation_vs_q4(
                    analysis, region, CONFIG
                )
            )
            no_hesitation_vs_q4_decoder_paths = (
                save_publication_figure(
                    no_hesitation_vs_q4_decoder_figure,
                    analysis,
                    region,
                    "svm_no_hesitation_vs_q4",
                    CONFIG,
                    analysis_path=(
                        "svm",
                        "no_hesitation_vs_q4",
                    ),
                )
            )
            figure_rows.extend(
                {
                    "eid": eid,
                    "region": region,
                    "analysis": "svm/no_hesitation_vs_q4",
                    "plot": "SVM — no hesitation vs Q4",
                    "phase": "All phases",
                    "file": str(path),
                }
                for path in no_hesitation_vs_q4_decoder_paths
            )
            plt.show()
            plt.close(no_hesitation_vs_q4_decoder_figure)

        if region in analysis.pca_results:
            for contrast in CONFIG.contrast_levels:
                contrast_percent = contrast * 100
                contrast_token = (
                    f"{contrast_percent:g}".replace(".", "p")
                )
                event_window_2d_figure, _ = plot_event_window_pca_2d(
                    analysis,
                    region,
                    contrast,
                    CONFIG,
                )
                event_window_2d_paths = save_publication_figure(
                    event_window_2d_figure,
                    analysis,
                    region,
                    (
                        "event_window_pca_2d_contrast_"
                        f"{contrast_token}"
                    ),
                    CONFIG,
                    analysis_path=("pca_event_windows", "2d"),
                )
                figure_rows.extend(
                    {
                        "eid": eid,
                        "region": region,
                        "analysis": "pca_event_windows/2d",
                        "plot": "Event-window PCA — 2D",
                        "phase": (
                            f"Contrast {contrast_percent:g}%"
                        ),
                        "file": str(path),
                    }
                    for path in event_window_2d_paths
                )
                plt.show()
                plt.close(event_window_2d_figure)

                event_window_3d_figure, _ = plot_event_window_pca_3d(
                    analysis,
                    region,
                    contrast,
                    CONFIG,
                )
                event_window_3d_paths = save_publication_figure(
                    event_window_3d_figure,
                    analysis,
                    region,
                    (
                        "event_window_pca_3d_contrast_"
                        f"{contrast_token}"
                    ),
                    CONFIG,
                    analysis_path=("pca_event_windows", "3d"),
                )
                figure_rows.extend(
                    {
                        "eid": eid,
                        "region": region,
                        "analysis": "pca_event_windows/3d",
                        "plot": "Event-window PCA — 3D",
                        "phase": (
                            f"Contrast {contrast_percent:g}%"
                        ),
                        "file": str(path),
                    }
                    for path in event_window_3d_paths
                )
                plt.show()
                plt.close(event_window_3d_figure)

        if region in analysis.continuous_pca_results:
            for contrast in CONFIG.contrast_levels:
                contrast_percent = contrast * 100
                contrast_token = (
                    f"{contrast_percent:g}".replace(".", "p")
                )
                continuous_2d_figure, _ = plot_continuous_pca_2d(
                    analysis,
                    region,
                    contrast,
                    CONFIG,
                )
                continuous_2d_paths = save_publication_figure(
                    continuous_2d_figure,
                    analysis,
                    region,
                    (
                        "continuous_pca_2d_contrast_"
                        f"{contrast_token}"
                    ),
                    CONFIG,
                    analysis_path=("pca_continuous", "2d"),
                )
                figure_rows.extend(
                    {
                        "eid": eid,
                        "region": region,
                        "analysis": "pca_continuous/2d",
                        "plot": "Continuous PCA — 2D",
                        "phase": (
                            f"Contrast {contrast_percent:g}%"
                        ),
                        "file": str(path),
                    }
                    for path in continuous_2d_paths
                )
                plt.show()
                plt.close(continuous_2d_figure)

                continuous_3d_figure, _ = plot_continuous_pca_3d(
                    analysis,
                    region,
                    contrast,
                    CONFIG,
                )
                continuous_3d_paths = save_publication_figure(
                    continuous_3d_figure,
                    analysis,
                    region,
                    (
                        "continuous_pca_3d_contrast_"
                        f"{contrast_token}"
                    ),
                    CONFIG,
                    analysis_path=("pca_continuous", "3d"),
                )
                figure_rows.extend(
                    {
                        "eid": eid,
                        "region": region,
                        "analysis": "pca_continuous/3d",
                        "plot": "Continuous PCA — 3D",
                        "phase": (
                            f"Contrast {contrast_percent:g}%"
                        ),
                        "file": str(path),
                    }
                    for path in continuous_3d_paths
                )
                plt.show()
                plt.close(continuous_3d_figure)

    diagnostic_frames = [
        frame.assign(source=name)
        for name, frame in [
            ("probe", analysis.probe_errors),
            ("peth", analysis.peth_errors),
            ("hesitation_peth", analysis.hesitation_peth_errors),
            (
                "hesitation_presence_peth",
                analysis.hesitation_presence_peth_errors,
            ),
            (
                "no_hesitation_vs_q4_peth",
                analysis.no_hesitation_vs_q4_peth_errors,
            ),
            ("decoder", analysis.decoder_errors),
            ("pca", analysis.pca_errors),
            ("continuous_pca", analysis.continuous_pca_errors),
        ]
        if not frame.empty
    ]
    if diagnostic_frames:
        print("Recorded diagnostics:")
        display(pd.concat(diagnostic_frames, ignore_index=True))

figure_manifest_df = pd.DataFrame(figure_rows)
if not figure_manifest_df.empty:
    display(figure_manifest_df)


## 14. Optional multi-session summary

For more than one EID, this table reports the mean session-level decoder performance by
region and phase. It does not pool neurons across animals and therefore avoids pretending
that different sessions share the same feature space.


In [ ]:
if decoder_summary_df.empty:
    print("No decoder summary is available.")
elif decoder_summary_df["eid"].nunique() == 1:
    print("Add more EIDs to SESSION_IDS for a multi-session aggregate.")
else:
    multisession_decoder_summary_df = (
        decoder_summary_df.groupby(["region", "phase"], as_index=False)
        .agg(
            n_sessions=("eid", "nunique"),
            mean_accuracy=("accuracy", "mean"),
            sem_accuracy=(
                "accuracy",
                lambda values: values.std(ddof=1) / np.sqrt(values.count()),
            ),
            mean_balanced_accuracy=("balanced_accuracy", "mean"),
        )
    )
    display(multisession_decoder_summary_df)


## 15. Download all generated files as one ZIP

Run this final cell in Google Colab after the analysis cells finish. It archives the
complete configured output directory, including every session and region subdirectory.


In [ ]:
def zip_and_download_output_directory(
    output_directory: str | Path,
    archive_name: str | None = None,
) -> Path:
    output_dir = Path(output_directory).expanduser().resolve()
    if not output_dir.is_dir():
        raise FileNotFoundError(
            f"Output directory does not exist: {output_dir}"
        )
    if not any(path.is_file() for path in output_dir.rglob("*")):
        raise RuntimeError(f"Output directory is empty: {output_dir}")

    archive_stem = archive_name or f"{output_dir.name}_all_outputs"
    archive_base = output_dir.parent / archive_stem
    archive_path = Path(
        shutil.make_archive(
            str(archive_base),
            "zip",
            root_dir=output_dir.parent,
            base_dir=output_dir.name,
        )
    )
    print(f"ZIP ready: {archive_path}")

    if colab_files is None:
        print("Automatic download is available in Google Colab.")
    else:
        colab_files.download(str(archive_path))

    return archive_path


ZIP_PATH = zip_and_download_output_directory(CONFIG.figure_output_dir)
